### setup

In [1]:
# !pip install -U sentence-transformers
# !pip install datasets
# !pip install tensorflow_ranking

In [2]:
import pickle
import numpy as np
import os
import tqdm
import torch
import gc

from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()

from importlib import reload
from matplotlib import pyplot as plt

import matrix_approx_zeshel

matrix_approx_zeshel = reload(matrix_approx_zeshel)

!ls  | grep matrix_approx_zeshel

2024-03-23 03:42:19.308350: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-03-23 03:42:19.759525: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-03-23 03:42:19.762848: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-03-23 03:42:21.606042: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


matrix_approx_zeshel.py


In [24]:
import collections
import pickle
import numpy as np
import tqdm
import os
import gc

import matplotlib.pyplot as plt


def load(limit, raw_path = "stand/log.local.logtime2.txt", path = "log.local.logtime2.bin", key_games = None, seed = 17, det_attempts = 0, key_size = 100):
    readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
    requests = list()
    docembs = collections.defaultdict(dict)

    if os.path.isfile(path):
        with open(path, "rb") as f:
            flimit, frequests, fdocembs = pickle.load(f)
            if flimit == limit:
                requests, docembs = frequests, fdocembs
            else:
                print(f"WARN: buffered limit is different, {flimit} != {limit}, reloading...")

    if not requests:
        with open(raw_path) as f:
            req = list()
            reqid = None
            models = list()
            prevreqmodel = None
            reqmodel = dict()
            prevmodelid = -1
            bannermodelid = -1
            for i, line in tqdm.tqdm_notebook(enumerate(f)):
                if line.startswith("Model = 6;"):
                    prevreqmodel = reqmodel
                    reqmodel = dict()

                if line.startswith("Model = "):
                    spl = line.split(" ")
                    prevmodelid = int(spl[2][:-1])
                    bannermodelid = max(bannermodelid , prevmodelid)
                    reqmodel[prevmodelid] = readvector(spl[3])
                elif line.startswith("dbid"):
                    spl = line.split(" ")
                    dbid = int(spl[1][:-1])
                    docembs[bannermodelid][dbid] = readvector(spl[2])
                elif line.startswith("seed"):
                    if len(requests) >= limit:
                        break
                    if req:
                        requests.append((reqid, prevreqmodel, sorted(req)))
                        req = list()
                    reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
                else:
                    req.append(
                        (int(line.split()[0]), float(line.split()[1]))
                    )
        
        with open(path, "wb") as f:
            pickle.dump((limit, requests, docembs), f)

    games_count = len(requests[0][2])
    assert games_count == 16514
    requests = [r for r in requests if len(r[2]) == games_count]
    
    print([(i, len(docembs[i].keys())) for i in docembs])  # should be equal
    docblocks = {
        mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
        for mid in docembs
    }
    
    class EvalContext:
        def __init__(self, games_count = games_count, key_size = key_size, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
            self.games_count = games_count
            
            self.key_size = key_size
            self.key_games = (
                np.random.choice(np.arange(games_count), key_size, replace=False)
                if key_games is None else
                key_games
            )

            self.requests = requests
            np.random.seed(seed)
            np.random.shuffle(self.requests)
            
            self.try_det_attempts(det_attempts)

            self.key_reqs = self.requests[:key_size + 1]
            self.key_reqs_idx = np.arange(key_size + 1)

            self.train_split = int(len(self.requests) * train_size)

            assert key_size + 1 < self.train_split

            self.train_reqs = self.requests[key_size + 1: self.train_split]
            self.test_reqs = self.requests[self.train_split:]

            self.slices = ["key", "train", "test"]
            print(len(self.key_reqs), len(self.train_reqs), len(self.test_reqs))

            self.docblocks = docblocks
            self.relevs = dict()
            
        def get_top_games(self):
            if not hasattr(self, "top_games"):
                embed_games = np.array([
                    np.array([r[2][g_i][1] for r in self.get_requests("train")])
                    for g_i in range(self.games_count)
                ])

                self.embed_games_mean = embed_games.mean(axis=1)
                self.top_games_all = (-self.embed_games_mean).argsort()
                self.top_games = self.top_games_all[:len(self.key_games)]

            return self.top_games
        
        def set_top_games_as_key(self):
            self.key_games = self.get_top_games()
            return self
        
        def get_kmeans_games(self, all_from_labels=True):
            X = self.get_relevs("train").T

            k_func = (
                (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
                if not all_from_labels else
                (lambda C : from_labels(X, C.labels_))
            )
            K_KMeans = k_func(
                KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
            )

            return K_KMeans

        def set_kmeans_games_as_key(self, *args, **kwargs):
            self.key_games = self.get_kmeans_games(*args, **kwargs)
            return self

        def try_det_attempts(self, det_attempts, model_id = 6):
            def get_det(r, r_i_array):
                kr = np.array([
                    r[r_i][1][model_id][:self.key_size]
                    for r_i in r_i_array[:self.key_size]
                ])
                return np.abs(np.linalg.det(kr[:kr.shape[1], :]))

            best_i_array = np.arange(len(self.requests))

            for _ in range(det_attempts):
                # print("try update key_reqs...")
                
                r_i_array = np.arange(len(self.requests))
                np.random.shuffle(r_i_array)
                
                n, o = get_det(self.requests, r_i_array), get_det(self.requests, best_i_array)
                # print(n, o)
                if n > o:
                    best_i_array = r_i_array
                    # print("updated!")

            print("Best det = ", get_det(self.requests, best_i_array))
            
            new_requests = [
                self.requests[i]
                for i in best_i_array
            ]
            
            del self.requests
            gc.collect()

            self.requests = new_requests
            print(get_det(self.requests, np.arange(len(self.requests))))

        def get_relevs(self, t = "train"):
            if t not in self.relevs:
                self.relevs[t] = np.array([
                    np.array([g_i[1] for g_i in r[2]])
                    for r in self.get_requests(t)
                ])
                
            return self.relevs[t]

        def get_requests(self, t = "train"):
            if t == "train":
                return self.train_reqs
            elif t == "key":
                return self.key_reqs
            elif t == "test":
                return self.test_reqs
            else:
                assert False

    return EvalContext(key_games = key_games, seed = seed, det_attempts = det_attempts)

In [17]:
def load_ment_to_ent_scores(directory = "yugioh", shuffle_rows = 0, full = True):
    data = list()

    for file in os.listdir(directory):
        path = f"{directory}/{file}"
        print(f"Loading file {path}")
        with open(path, "rb") as f:
            data.append(
                pickle.load(f)
            )
    data = sorted(data, key = lambda x: x["arg_dict"]["n_ment_start"])

    for i in range(len(data) - 1):
        if full:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] == data[i + 1]["arg_dict"]["n_ment_start"]
        else:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] <= data[i + 1]["arg_dict"]["n_ment_start"]
        
    ment_to_ent_scores = list(map(lambda x: x["ment_to_ent_scores"], data))
    ment_to_ent_scores = np.vstack(ment_to_ent_scores)
    print("Loaded shape = ", ment_to_ent_scores.shape)
    
    if shuffle_rows:
        print(f"Shuffling... (seed = {shuffle_rows})")
        np.random.seed(shuffle_rows)
        np.random.shuffle(ment_to_ent_scores)
    
    return ment_to_ent_scores

In [38]:
from sklearn.cluster import KMeans
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances


def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

class EvalContextRelevs:
    def __init__(self, relevs, key_size = 100, train_size = 0.7, key_games = None, seed = 17, shuffle=False, det_attempts = 0):
        self.relevs = np.array(relevs)
        self.reqs_count = self.relevs.shape[0]
        self.games_count = self.relevs.shape[1]
        
        self.key_size = key_size
        self.key_games = (
            np.random.choice(np.arange(self.games_count), key_size, replace=False)
            if key_games is None else
            key_games
        )
        np.random.seed(seed)
        if shuffle:
            np.random.shuffle(self.relevs)

        self.try_det_attempts(det_attempts)
        self.train_split = int(self.reqs_count * train_size)

        assert key_size + 1 < self.train_split


        self.key_relevs = self.relevs[:key_size]
        self.train_relevs = self.relevs[key_size + 1: self.train_split]
        self.test_relevs = self.relevs[self.train_split:]

        self.slices = ["key", "train", "test"]
        print(len(self.key_relevs), len(self.train_relevs), len(self.test_relevs))

    def get_top_games(self):
        return self.relevs.mean(axis=0).argsort()[:self.key_size]

    def set_top_games_as_key(self):
        self.key_games = self.get_top_games()
        return self

    def get_kmeans_games(self, all_from_labels=True):
        X = self.get_relevs("train").T

        k_func = (
            (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
            if not all_from_labels else
            (lambda C : from_labels(X, C.labels_))
        )
        K_KMeans = k_func(
            KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
        )
        
        return K_KMeans
    
    def set_kmeans_games_as_key(self, *args, **kwargs):
        self.key_games = self.get_kmeans_games(*args, **kwargs)
        return self
    

    def try_det_attempts(self, det_attempts):
        def get_det(r):
            kr = r[:self.key_size, self.key_games] - r.mean()
            return np.abs(np.linalg.det(kr))

        best_i_array = np.arange(len(self.relevs))

        for i in range(det_attempts):

            r_i_array = np.arange(len(self.relevs))
            np.random.shuffle(r_i_array)

            n, o = get_det(self.relevs[r_i_array, :]), get_det(self.relevs[best_i_array, :])
            
            # print(f"try update key_reqs ({o} vs {n}...")
            if n > o:
                best_i_array = r_i_array
                print(f"updated det ({i}, {o} -> {n})")

        print("Best det = ", get_det(self.relevs[best_i_array, :]))

        self.relevs = self.relevs[best_i_array, :]
        print("Current de = ", get_det(self.relevs))
        
    def get_relevs(self, t = "train"):
        if t == "train":
            return self.train_relevs
        elif t == "key":
            return self.key_relevs
        elif t == "test":
            return self.test_relevs
        else:
            assert False
            
    def get_requests(self, t = "train"):
        if t == "train":
            return self.train_reqs
        elif t == "key":
            return self.key_reqs
        elif t == "test":
            return self.test_reqs
        else:
            assert False

In [19]:
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        self.trus_top = dict()
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        return [self.top_logits for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):    
        recs = self.recommend(t)
        
        if isinstance(recs, list):
            recs = np.array(recs)
        
        mse = np.mean((recs - self.get_user_scores(t)) ** 2)
        
        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :topsize].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :topsize]
        
        if (t, topsize) not in self.trus_top:
            trus = self.get_user_scores(t)
            trus = np.argsort(-trus, axis=1)[:, :topsize]
            self.trus_top[(t, topsize)] = trus
            
        trus = self.trus_top[(t, topsize)]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)
    
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

DEFAULT_FIT_KWARGS = {
    "learning_rate": 0.001,
    "n": 500,
    "c": 5000,
    "matrix_l2": 0,
    "dssm_l2": 0,
    "train_popbias": False,
    "train_bias": False,
    "train_vbias": False,
    "verbose": True,
    "train_matrix": True,
    "train_dssm": False,
    "loss": "mse",
    "ubatch": 1e9,
    "activation": "relu",
    "score_verbose": 0,
    "trainable_items": False,
    "use_keys_in_train": False,
    "Winit": "glorot0.01",
    "Wfreeze": False,
    "TEinit": "legacy",
    "normalize_embs": True,
    "save_callback": None
}

class CBKnnV0(Popular):
    def __init__(self, ctx, fit_kwargs = dict()):
        super().__init__(ctx)
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(fit_kwargs)
        self.fit_kwargs = p
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        
        self.embed_users = {
            t : np.array([
                    np.array([r_i[i] for i in sorted(ctx.key_games)])
                    for r_i in ctx.get_relevs(t)
                ])
            for t in ctx.slices
        }
        self.embed_users_mean = {
            t : self.embed_users[t].mean(axis = 0)
            for t in ctx.slices
        }
        # embed_users = embed_users - embed_users_mean
        print("self.embed_users['train'].shape = ", self.embed_users['train'].shape )
        
        self.embed_games = np.array([
            np.array([r[g_i] for r in ctx.get_relevs("key")])
            for g_i in range(ctx.games_count)
        ])
        
        self.embed_games_mean = self.embed_games.mean(axis = 0)
        
        # embed_games = embed_games - embed_games_mean
        print("self.embed_games.shape = ", self.embed_games.shape)
        
        
        if self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["TEinit"] == "legacy":
                value = self.embed_games
            elif self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                value = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print(type(value))
            else:
                assert False, "unknown TEinit: " + str(self.fit_kwargs["TEinit"])
                
            self.trainable_games = tf.Variable(
                tf.convert_to_tensor(value, dtype=tf.float32),
                trainable=True
            )
        else:
            if self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                self.embed_games = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print("ANNCur init")
            
        
        self.games_top_key = self.embed_games.mean(axis = 1)
        
        self.games2users = np.array([
            self.embed_games[g_i]
            for g_i in ctx.key_games
        ])
        print("self.games2users.shape = ", self.games2users.shape)
        
        self.core_users_scores = ctx.get_relevs("key")
        print("self.core_users_scores.shape = ", self.core_users_scores.shape)
        
        self.core_users_embs = self.embed_users["key"]
        print("self.core_users_embs.shape = ", self.core_users_embs.shape)
        
        self.ge_users = (
            (self.embed_users["train"].T / self.embed_users["train"].mean(axis=1)).T @ self.games2users
        )
        # ge_users = embed_users @ games2users
        print("self.ge_users.shape = ", self.ge_users.shape)
        
        self.slice2best = {
            t : 0
            for t in ctx.slices
        }
    
    def __repr__(self):
        return "CbKnn(" + str(self.fit_kwargs) + ")"
        
    # inherited   
    # def get_user_scores(self, t):
    
    def get_user_embs(self, t):
        if self.fit_kwargs["normalize_embs"]:
            return (self.embed_users[t] - self.embed_users_mean["train"]) / self.embed_users[t].std(axis=0)
        else:
            return self.embed_users[t]
    
    def get_game_embs(self):
        if not self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["normalize_embs"]:
                return (self.embed_games - self.embed_games.mean(axis=0)) / self.embed_games.std(axis=0)
            else:
                return self.embed_games 
        else:
            return self.trainable_games

    def fit(self, **kwargs):
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        p.update(kwargs)

        self.p = p
        self.train_bias = p["train_bias"]
        self.train_vbias = p["train_vbias"]
        self.train_popbias = p["train_popbias"]
        self.train_matrix = p["train_matrix"]
        self.train_dssm = p["train_dssm"]
        self.loss = p["loss"]

        if p["use_keys_in_train"]:
            train_user_scores = np.vstack([
                self.get_user_scores("key"),
                self.get_user_scores("train")
            ])
            train_user_embs = np.vstack([
                self.get_user_embs("key"),
                self.get_user_embs("train")
            ])
        else:
            train_user_scores = self.get_user_scores("train")
            train_user_embs = self.get_user_embs("train")

        game_embs = self.get_game_embs()
        
        
        
        if self.p["Winit"] == "glorot0.01":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
            values = values / 100.  
        elif self.p["Winit"] == "glorot":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        elif self.p["Winit"] == "eye":
            values = np.eye(train_user_embs.shape[1], game_embs.shape[1])
        else:
            assert False, "init is not known:" + str(self.p["init"])
            
        self.W = tf.Variable(values, trainable = True) 
        self.pb = tf.Variable(1., trainable = True) 
        self.b = tf.Variable(0., trainable = True) 
        
        if p["verbose"]:
            print("self.W = ", self.W)
            print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        opt =  tf.keras.optimizers.Adam(learning_rate=p["learning_rate"])

        if self.train_dssm:
            self.train_dssm = True

            dim = game_embs.shape[1]
            self.g_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.g_dssm(game_embs)
            
            # dim = train_user_embs.shape[1]
            self.u_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.u_dssm(train_user_embs)
            
        if self.train_vbias:
            self.vb = tf.Variable(
                np.zeros_like(self.game_avg_scores["train"], dtype=np.float32),
                trainable = True
            ) 
        
        
        for i in range(p["n"]):
            def loss():
                def get_logits_scores(loss_batch = 1e9):
                    game_slice = None
                    
                    ubatch = p["ubatch"]
                    if ubatch >= train_user_scores.shape[0]:
                        train_user_embs_ = train_user_embs
                        scores_ = train_user_scores
                    else:
                        u_slice = np.random.choice(np.arange(train_user_scores.shape[0]), ubatch, replace = True)
                        train_user_embs_ = train_user_embs[u_slice]
                        scores_ = train_user_scores[u_slice]
                    
                    if loss_batch >= game_embs.shape[0]:
                        game_embs_ = game_embs
                        scores_ = scores_
                    else:
                        game_slice = np.random.choice(np.arange(game_embs.shape[0]), loss_batch, replace = True)
                        game_embs_ = (
                            game_embs[game_slice]
                            if isinstance(game_embs, np.ndarray) else
                            tf.gather(game_embs, game_slice)
                        )
                        scores_ = scores_[:, game_slice]
                        
                    
                    logits = 0
                    
                    if self.train_matrix:
                        logits += train_user_embs_ @ self.W @ tf.transpose(game_embs_)

                    if self.train_dssm:
                        logits += self.u_dssm(train_user_embs_) @ tf.transpose(self.g_dssm(game_embs_))
                    
                    if self.train_popbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.pb * self.game_avg_scores["train"]
                        else:
                            logits += self.pb * self.game_avg_scores["train"][game_slice]
                        
                    if self.train_vbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.vb
                        else:
                            logits += tf.gather(self.vb, game_slice)
                        
                    if self.train_bias:
                        logits += self.b
                        
                    return logits, scores_
                        
                if self.loss == "mse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    v = tf.reduce_mean((scores - logits) ** 2)
                elif self.loss == "qmse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    q_mean = scores.mean(axis=1)
                    v = tf.reduce_mean(((scores.T - q_mean).T - logits) ** 2)
                elif self.loss == "ApproxNDCGLoss":
                    while True:
                        logits, scores = get_logits_scores(
                            loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))

                        v = tfr.keras.losses.ApproxNDCGLoss()(
                            scores.astype(np.float32),
                            logits
                        )
                    
                        if not tf.math.is_nan(v).numpy().any():
                            break
                        else:
                            if p["verbose"]:
                                print("nanloss ignored")
                elif self.loss == "softmax":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ))
                elif self.loss == "softmax_signed":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    sf = tf.nn.softmax(logits, axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        sf,
                        -sf
                    ))
                elif self.loss == "softmax_weighted":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ) * scores)
                elif self.loss == "sigmoid_top_100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    mask = np.argsort(-scores, axis=1) < 100
                    v = -tf.reduce_sum(
                        tf.math.log_sigmoid((2 * mask - 1) * logits)
                    )
                elif self.loss == "KL100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    mask = np.argsort(-scores, axis=1) < 100
                    v = tf.keras.losses.KLDivergence()((scores.T >= qs.T).T, logits)
                else:
                    assert False
                    
                if self.p["c"]:
                    v += tf.reduce_mean(self.W * self.W) * p["c"]
                    
                if self.p["dssm_l2"]:
                    for weights_ in [self.u_dssm.weights, self.g_dssm.weights]:
                        for weight_ in weights_:
                            v += tf.reduce_sum(weight_ * weight_) * self.p["dssm_l2"]
                
                if self.p["matrix_l2"]:
                    v += tf.reduce_sum(self.W * self.W) * self.p["matrix_l2"]
                
                if p["verbose"]:
                    print(v.numpy())
                
                return v

            weights = list()
            
            if self.train_matrix and (not self.p["Wfreeze"]):
                weights += [self.W]

            if self.train_dssm:
                weights += self.u_dssm.weights
                weights += self.g_dssm.weights
            
            if self.train_popbias:
                weights += [self.pb]

            if self.train_vbias:
                weights += [self.vb]   
                
            if self.train_bias:
                weights += [self.b]   
                
            
            if self.p["trainable_items"]:
                weights += [self.trainable_games]
                
                
            opt.minimize(loss, weights)
            
            if p["score_verbose"] and (i % p["score_verbose"] == 0):
                print(f"\n=== Iteration {i} ===")
                score = dict()
                for sl in self.ctx.slices:
                    score[sl] = self.get_score(sl)
                    print(f"slice = {sl}, score = {score[sl]}")
                    
                if score["train"] > self.slice2best["train"]:
                    self.slice2best = score
                    if p["save_callback"] is not None:
                        p["save_callback"](self)
                
        print("last loss = ", loss())

    def recommend(self, t):
        logits = 0
                    
        if self.train_matrix:
            logits += self.get_user_embs(t) @ self.W @ tf.transpose(self.get_game_embs())

        if self.train_dssm:
            logits += self.u_dssm(self.get_user_embs(t)) @ tf.transpose(self.g_dssm(self.get_game_embs()))

        if self.train_popbias:
            logits += self.pb * self.game_avg_scores["train"]       
            
        if self.train_vbias:
            logits += self.vb
            
        if self.train_bias:
            logits += self.b
            
        return logits
    
    # inherited
    # def get_score(self, t, topsize = 100):
    
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))
            
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))
            
class AnnCUR(Popular):
    def __init__(self, ctx, oracle=False, key_games=None, name=None):
        super().__init__(ctx)
        self.oracle = oracle
        self.ctx = ctx
        self.name = name

        self.key_cols_idx = np.array(sorted(ctx.key_games if key_games is None else key_games))
        rows_idx = np.arange(ctx.get_relevs("train").shape[0])
        
        self.cur = matrix_approx_zeshel.CURApprox(
            rows=torch.from_numpy(ctx.get_relevs("train")),
            cols=torch.from_numpy(ctx.get_relevs("train")[:, self.key_cols_idx]),
            row_idxs=rows_idx,
            col_idxs=self.key_cols_idx,
            approx_preference="rows",
            A=(torch.from_numpy(ctx.get_relevs("train")) if oracle else None)
        )        
        
    def __repr__(self):
        if self.name is None:
            return f"AnnCur({id(self)})"
        else:
            return f"AnnCur({self.name} - {id(self)})"
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        return self.cur.latent_cols.T

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        key_relevs = torch.from_numpy(self.ctx.get_relevs(t)[:, self.key_cols_idx])
        return self.cur.get_complete_row(key_relevs)
    
    def get_score(self, t, topsize = 100):       
        recs = np.array(self.recommend(t))
        trus = self.get_user_scores(t)

        mse = np.mean((recs - trus) ** 2)

        recs = np.argsort(-recs, axis=1)[:, :topsize]
        trus = np.argsort(-trus, axis=1)[:, :topsize]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

In [20]:
def coitem_algorithm(n_support, candidate_items, target_items, min_item_rel_norm=1e-5, eps=1e-6):
    support_items = []
    support_items_scores = []
    n_c, n_q = candidate_items.shape
    n_t = target_items.shape[0]

    candidate_item_squared_norms = (candidate_items ** 2).sum(axis=1)
    min_item_norm = min_item_rel_norm * np.sqrt(candidate_item_squared_norms.mean())
    
    candidate_coitems = candidate_items.dot(
        target_items.T.dot(target_items)
    )
    orthonormed_support_items = np.zeros((n_support, n_q))
    remaining_items = np.ones(candidate_items.shape[0], dtype="bool")
    for t in tqdm.tqdm_notebook(range(n_support)):
        scores = (candidate_coitems * candidate_items).sum(axis=1)
        remaining_items &= (candidate_item_squared_norms >= min_item_norm ** 2)
        scores[remaining_items] /= candidate_item_squared_norms[remaining_items]
        scores[~remaining_items] = 0
        
        new_item_id = np.argmax(scores)
        assert remaining_items[new_item_id]
        support_items.append(new_item_id)
        support_items_scores.append(scores[new_item_id] / (n_t * n_q))
        remaining_items[new_item_id] = False
        
        new_item = candidate_items[new_item_id].copy()
        new_item -= orthonormed_support_items[:t].T.dot(
            orthonormed_support_items[:t].dot(new_item)
        )
        assert np.allclose((new_item ** 2).sum(), candidate_item_squared_norms[new_item_id], atol=eps)
        new_item /= np.sqrt((new_item ** 2).sum())
        orthonormed_support_items[t] = new_item
        new_coitem = candidate_coitems[new_item_id] / np.sqrt(candidate_item_squared_norms[new_item_id])
        
        coefs = (candidate_items * new_item).sum(axis=1)
        candidate_item_squared_norms -= coefs ** 2
        assert np.all(candidate_item_squared_norms[remaining_items] > -eps)
        
        candidate_coitems -= coefs.reshape((-1, 1)).dot(new_coitem.reshape((1, -1)))
        cocoefs = (candidate_coitems * new_item).sum(axis=1, keepdims=True)
        candidate_coitems -= cocoefs.dot(new_item.reshape((1, -1)))
    return np.array(support_items, dtype=np.int64), np.array(support_items_scores)

# Preparate

In [21]:
def do(ctx, name, coitem=True, kmeans=True, top=True, random=True):
    if coitem:
        t = ctx.get_relevs("train").T
        t = (t - t.mean()) / t.std()  # like in other comparisons with RBE
        ctx.key_games = list(coitem_algorithm(ctx.key_size, t, t, 1e-8, eps=1e9)[0])

        ma = AnnCUR(ctx, key_games=ctx.key_games)

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")

        GE = ma.get_game_embs()
        QE = torch.from_numpy(ma.ctx.get_relevs("test")[:, ma.key_cols_idx])

        fname = f"./GE_QE_AnnCURxCoItem_{name}_{str(round(te, 4))[2:]}.npz"
        np.savez_compressed(fname, QE=QE.numpy(), GE=GE.numpy())
        print(fname)
        
    if kmeans:
        ctx.set_kmeans_games_as_key()
        ma = AnnCUR(ctx, key_games=ctx.key_games)

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")

        GE = ma.get_game_embs()
        QE = torch.from_numpy(ma.ctx.get_relevs("test")[:, ma.key_cols_idx])

        fname = f"./GE_QE_AnnCURxKMeans_{name}_{str(round(te, 4))[2:]}.npz"
        np.savez_compressed(fname, QE=QE.numpy(), GE=GE.numpy())
        print(fname)
        
    if top:
        ctx.set_top_games_as_key()
        ma = AnnCUR(ctx, key_games=ctx.key_games)

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")

        GE = ma.get_game_embs()
        QE = torch.from_numpy(ma.ctx.get_relevs("test")[:, ma.key_cols_idx])

        fname = f"./GE_QE_AnnCURxTop_{name}_{str(round(te, 4))[2:]}.npz"
        np.savez_compressed(fname, QE=QE.numpy(), GE=GE.numpy())
        print(fname)
        
    if random:
        ctx.key_games = np.random.choice(np.arange(ctx.games_count), ctx.key_size, replace=False)
        ma = AnnCUR(ctx, key_games=ctx.key_games)

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")

        GE = ma.get_game_embs()
        QE = torch.from_numpy(ma.ctx.get_relevs("test")[:, ma.key_cols_idx])

        fname = f"./GE_QE_AnnCURxRandom_{name}_{str(round(te, 4))[2:]}.npz"
        np.savez_compressed(fname, QE=QE.numpy(), GE=GE.numpy())
        print(fname)

In [30]:
m_ = None
def do_RBE(ctx, name, coitem=True, kmeans=True, top=True, random=True, N = 100000, LR = 0.0005):
    def get_save_callback(t, fname):
        def save_callback(m):
            global m_
            m_ = m
            GE_ = m.get_game_embs()
            GE = np.hstack([
                GE_,
                m.g_dssm(GE_),
                m.vb.numpy().reshape((-1, 1)) ,  # Vertical bias
                ctx.get_relevs("train").mean(axis=0).reshape((-1, 1))  # popbias
            ])

            R_All = ctx.get_relevs(t)
            QE_ = m.get_user_embs(t)
            QE = np.hstack([
                QE_ @ m.W,
                m.u_dssm(QE_),
                np.ones((R_All.shape[0], 1)),
                np.ones((R_All.shape[0], 1)) * m.pb
            ])
            
            l, r = fname.split(".npz")
            score = m.get_score(t)
            fname_ = l + f"_{str(round(score, 4))}.npz" + r
            print(f"Saving {fname_} ...")
            np.savez_compressed(fname_, QE=QE, GE=GE)
        return save_callback
            
    if coitem:
        t = ctx.get_relevs("train").T
        t = (t - t.mean()) / t.std()  # like in other comparisons with RBE
        ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

        ma = CBKnnV0(ctx, fit_kwargs={
            'c': 0,
            'train_matrix': True,
            'train_dssm': True,
            'train_vbias': True,
            'train_popbias': True, 'train_bias': True,
            'verbose': False, 'loss': 'softmax_signed',
            'loss_batch': 128, 'loss_q': 0.99,
            # 'dssm_l2': 5e-5,
            'activation': 'elu',
            'n': N,
            # 'ubatch': 1500,
            'score_verbose': 1000,
            'trainable_items': False,
            "TEinit": "anncur",
            "use_keys_in_train": True, # <<< DIFF HERE
            "learning_rate": LR,
            "save_callback": get_save_callback("test", f"./GE_QE_RBExCoItem_{name}.npz")
        })

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")
        print(tr, te)
        
    if kmeans:
        ctx.set_kmeans_games_as_key()
        ma = CBKnnV0(ctx, fit_kwargs={
            'c': 0,
            'train_matrix': True,
            'train_dssm': True,
            'train_vbias': True,
            'train_popbias': True, 'train_bias': True,
            'verbose': False, 'loss': 'softmax_signed',
            'loss_batch': 128, 'loss_q': 0.99,
            # 'dssm_l2': 5e-5,
            'activation': 'elu',
            'n': N,
            # 'ubatch': 1500,
            'score_verbose': 1000,
            'trainable_items': False,
            "TEinit": "anncur",
            "use_keys_in_train": True, # <<< DIFF HERE
            "learning_rate": LR,
            "save_callback": get_save_callback("test", f"./GE_QE_RBExKMeans_{name}.npz")
        })

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")
        print(tr, te)
        
    if top:
        ctx.set_top_games_as_key()
        ma = CBKnnV0(ctx, fit_kwargs={
            'c': 0,
            'train_matrix': True,
            'train_dssm': True,
            'train_vbias': True,
            'train_popbias': True, 'train_bias': True,
            'verbose': False, 'loss': 'softmax_signed',
            'loss_batch': 128, 'loss_q': 0.99,
            # 'dssm_l2': 5e-5,
            'activation': 'elu',
            'n': N,
            # 'ubatch': 1500,
            'score_verbose': 1000,
            'trainable_items': False,
            "TEinit": "anncur",
            "use_keys_in_train": True, # <<< DIFF HERE
            "learning_rate": LR,
            "save_callback": get_save_callback("test", f"./GE_QE_RBExTop_{name}.npz")
        })

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")
        print(tr, te)
        
    if random:
        ma = CBKnnV0(ctx, fit_kwargs={
            'c': 0,
            'train_matrix': True,
            'train_dssm': True,
            'train_vbias': True,
            'train_popbias': True, 'train_bias': True,
            'verbose': False, 'loss': 'softmax_signed',
            'loss_batch': 128, 'loss_q': 0.99,
            # 'dssm_l2': 5e-5,
            'activation': 'elu',
            'n': N,
            # 'ubatch': 1500,
            'score_verbose': 1000,
            'trainable_items': False,
            "TEinit": "anncur",
            "use_keys_in_train": True, # <<< DIFF HERE
            "learning_rate": LR,
            "save_callback": get_save_callback("test", f"./GE_QE_{name}_RBExRandom.npz")
        })

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")
        print(tr, te)

In [22]:
def get_Rp_basic(GE, A, R, QE_i, L):
    Ae = GE[A]

    u = np.linalg.pinv(Ae.T @ Ae) @ Ae.T @ R
    assert u.shape == QE_i.shape
    Rp = GE @ (u * L + QE_i * (1 - L))
    
    return Rp


def do_AXN(ctx, loaded, get_Rp, Z="test", STRIP = 0.05, Ls=np.linspace(0, 1, 11), Ks=[25], T_TS_s = [(200, 100)], not_deduplicate = False):
    # q = ctx.get_requests(Z)
    R_All = ctx.get_relevs(Z)

    GE, QE = loaded["GE"], loaded["QE"]

    best = 0
    best_arg = None

    for randomFirst in [False]:
        for K in Ks:
            for L in Ls:
                for T, TS in T_TS_s:
                    results = list()


                    As = list()
                    for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                        A = (
                            ctx.key_games[:K]
                            if randomFirst else
                            (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                        )
                        Rt = R_All[i]
                        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                        for _ in range(T // K - 1):
                            Rp = get_Rp(GE, A, Rt[A], QE[i], L)

                            An = list(A)
                            for ai in Rp.argsort()[::-1]:
                                if not_deduplicate or (ai not in An):
                                    An.append(ai)
                                if len(An) == len(A) + K:
                                    break
                            A = np.array(An)

                        assert len(A) == T
                        As.append(A)


                        A = sorted([
                            (-Rt[ai], ai)
                            for ai in A
                        ])[:TS]
                        A = [ai for _, ai in A]

                        pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                        results.append(ev(pred_top, gt_top) / float(TS))

                    if np.mean(results) > best:
                        best = np.mean(results)
                        best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                        best_arg_d = {
                            "K": K,
                            "L": L,
                            "randomFirst": randomFirst,
                            "score": np.mean(results)
                        }
                    print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)} best = {best}")

    return best_arg

## RecSys LT

In [25]:
L = 7000
N = 1000
DA = 50

In [36]:
ctx = load(L, raw_path = "stand/log.local.logtime2.txt",
           path="log.local.logtime2.true.bin", seed=17, det_attempts=DA, key_size=50)

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.6517584969541223e-46
2.6517584969541223e-46
51 4819 2088


In [27]:
do(ctx, "RecSysLT_Key50_")

/tmp/ipykernel_965688/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/50 [00:00<?, ?it/s]

np.mean(results), mse, len(results) =  0.6059846441170367 0.17983846265287565 4819
np.mean(results), mse, len(results) =  0.6003927203065134 0.18862063249657898 2088
./GE_QE_AnnCURxCoItem_RecSysLT_Key50__6004.npz


/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


np.mean(results), mse, len(results) =  0.6156837518157294 0.16532563696577035 4819
np.mean(results), mse, len(results) =  0.6063218390804598 0.17790229603513558 2088
./GE_QE_AnnCURxKMeans_RecSysLT_Key50__6063.npz
np.mean(results), mse, len(results) =  0.6789333886698484 0.24656882605158767 4819
np.mean(results), mse, len(results) =  0.6686733716475096 0.2736024041926458 2088
./GE_QE_AnnCURxTop_RecSysLT_Key50__6687.npz
np.mean(results), mse, len(results) =  0.5453870097530608 0.22690394570700045 4819
np.mean(results), mse, len(results) =  0.5418247126436782 0.23091893566521546 2088
./GE_QE_AnnCURxRandom_RecSysLT_Key50__5418.npz


In [25]:
!ls | grep "GE.*RecSysLT_"

GE_QE_AnnCURxCoItem_RecSysLT_Round10_6565.npz
GE_QE_AnnCURxKMeans_RecSysLT_Round10_613.npz
GE_QE_AnnCURxRandom_RecSysLT_Round10_5842.npz
GE_QE_AnnCURxTop_RecSysLT_Round10_6695.npz


In [37]:
do_RBE(ctx, "RecSys_LT_Key50_")

/tmp/ipykernel_965688/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

self.embed_users['train'].shape =  (4819, 100)
self.embed_games.shape =  (16514, 51)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (51, 16514)
self.core_users_embs.shape =  (51, 100)
self.ge_users.shape =  (4819, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.12686274509803921 tf.Tensor(33.98851382733185, shape=(), dtype=float64) 51
slice = key, score = 0.12686274509803921
np.mean(results), mse, len(results) =  0.12324756173479977 tf.Tensor(34.64435745070774, shape=(), dtype=float64) 4819
slice = train, score = 0.12324756173479977
np.mean(results), mse, len(results) =  0.1221072796934866 tf.Tensor(34.64254095312705, shape=(), dtype=float64) 2088
slice = test, score = 0.1221072796934866
np.mean(results), mse, len(results) =  0.1221072796934866 tf.Tensor(34.64254095312705, shape=(), dtype=float64) 2088
Saving ./GE_QE_RBExCoItem_RecSys_LT_Key50__0.1221.npz ...

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.4786274509

np.mean(results), mse, len(results) =  0.6155696202531645 tf.Tensor(889.8420149213775, shape=(), dtype=float64) 4819
slice = train, score = 0.6155696202531645
np.mean(results), mse, len(results) =  0.6124760536398467 tf.Tensor(887.9354708497567, shape=(), dtype=float64) 2088
slice = test, score = 0.6124760536398467

=== Iteration 13000 ===
np.mean(results), mse, len(results) =  0.5933333333333333 tf.Tensor(1041.5456532175276, shape=(), dtype=float64) 51
slice = key, score = 0.5933333333333333
np.mean(results), mse, len(results) =  0.628165594521685 tf.Tensor(1029.5627918212103, shape=(), dtype=float64) 4819
slice = train, score = 0.628165594521685
np.mean(results), mse, len(results) =  0.6239942528735632 tf.Tensor(1027.010586261473, shape=(), dtype=float64) 2088
slice = test, score = 0.6239942528735632
np.mean(results), mse, len(results) =  0.6239942528735632 tf.Tensor(1027.010586261473, shape=(), dtype=float64) 2088
Saving ./GE_QE_RBExCoItem_RecSys_LT_Key50__0.624.npz ...

=== Iterati

np.mean(results), mse, len(results) =  0.6331465517241379 tf.Tensor(1791.119731659053, shape=(), dtype=float64) 2088
Saving ./GE_QE_RBExCoItem_RecSys_LT_Key50__0.6331.npz ...

=== Iteration 27000 ===
np.mean(results), mse, len(results) =  0.6049019607843137 tf.Tensor(1878.5077810265627, shape=(), dtype=float64) 51
slice = key, score = 0.6049019607843137
np.mean(results), mse, len(results) =  0.6415272878190496 tf.Tensor(1906.1297936562676, shape=(), dtype=float64) 4819
slice = train, score = 0.6415272878190496
np.mean(results), mse, len(results) =  0.6370402298850575 tf.Tensor(1899.1680683805555, shape=(), dtype=float64) 2088
slice = test, score = 0.6370402298850575
np.mean(results), mse, len(results) =  0.6370402298850575 tf.Tensor(1899.1680683805555, shape=(), dtype=float64) 2088
Saving ./GE_QE_RBExCoItem_RecSys_LT_Key50__0.637.npz ...

=== Iteration 28000 ===
np.mean(results), mse, len(results) =  0.6066666666666666 tf.Tensor(1947.0616750859058, shape=(), dtype=float64) 51
slice = k

...

np.mean(results), mse, len(results) =  0.6540796845818634 tf.Tensor(2120.317607025173, shape=(), dtype=float64) 4819
slice = train, score = 0.6540796845818634
np.mean(results), mse, len(results) =  0.6456944444444445 tf.Tensor(2114.0064541147667, shape=(), dtype=float64) 2088
slice = test, score = 0.6456944444444445
np.mean(results), mse, len(results) =  0.6456944444444445 tf.Tensor(2114.0064541147667, shape=(), dtype=float64) 2088
Saving ./GE_QE_RecSys_LT_Key50__RBExRandom_0.6457.npz ...

=== Iteration 39000 ===
np.mean(results), mse, len(results) =  0.6198039215686274 tf.Tensor(2240.8758943296207, shape=(), dtype=float64) 51
slice = key, score = 0.6198039215686274
np.mean(results), mse, len(results) =  0.6521456733762191 tf.Tensor(2187.5425723255294, shape=(), dtype=float64) 4819
slice = train, score = 0.6521456733762191
np.mean(results), mse, len(results) =  0.6433716475095786 tf.Tensor(2184.8275524728647, shape=(), dtype=float64) 2088
slice = test, score = 0.6433716475095786

=== I

np.mean(results), mse, len(results) =  0.6601182818012035 tf.Tensor(2662.942128987788, shape=(), dtype=float64) 4819
slice = train, score = 0.6601182818012035
np.mean(results), mse, len(results) =  0.650205938697318 tf.Tensor(2656.301263877692, shape=(), dtype=float64) 2088
slice = test, score = 0.650205938697318

=== Iteration 54000 ===
np.mean(results), mse, len(results) =  0.6347058823529411 tf.Tensor(2775.168400418106, shape=(), dtype=float64) 51
slice = key, score = 0.6347058823529411
np.mean(results), mse, len(results) =  0.6638493463374144 tf.Tensor(2716.95855408496, shape=(), dtype=float64) 4819
slice = train, score = 0.6638493463374144
np.mean(results), mse, len(results) =  0.6530890804597702 tf.Tensor(2713.7368109276745, shape=(), dtype=float64) 2088
slice = test, score = 0.6530890804597702
np.mean(results), mse, len(results) =  0.6530890804597702 tf.Tensor(2713.7368109276745, shape=(), dtype=float64) 2088
Saving ./GE_QE_RecSys_LT_Key50__RBExRandom_0.6531.npz ...

=== Iterati

np.mean(results), mse, len(results) =  0.6602873563218391 tf.Tensor(3028.4939044152597, shape=(), dtype=float64) 2088
slice = test, score = 0.6602873563218391
np.mean(results), mse, len(results) =  0.6602873563218391 tf.Tensor(3028.4939044152597, shape=(), dtype=float64) 2088
Saving ./GE_QE_RecSys_LT_Key50__RBExRandom_0.6603.npz ...

=== Iteration 69000 ===
np.mean(results), mse, len(results) =  0.6319607843137256 tf.Tensor(3157.604333803599, shape=(), dtype=float64) 51
slice = key, score = 0.6319607843137256
np.mean(results), mse, len(results) =  0.667159161651795 tf.Tensor(3103.1205563456374, shape=(), dtype=float64) 4819
slice = train, score = 0.667159161651795
np.mean(results), mse, len(results) =  0.6557040229885057 tf.Tensor(3099.1398683969655, shape=(), dtype=float64) 2088
slice = test, score = 0.6557040229885057

=== Iteration 70000 ===
np.mean(results), mse, len(results) =  0.6303921568627451 tf.Tensor(3176.036637248783, shape=(), dtype=float64) 51
slice = key, score = 0.63039

np.mean(results), mse, len(results) =  0.6607615687902054 tf.Tensor(3511.936086229303, shape=(), dtype=float64) 4819
slice = train, score = 0.6607615687902054
np.mean(results), mse, len(results) =  0.6498132183908045 tf.Tensor(3508.5301959639764, shape=(), dtype=float64) 2088
slice = test, score = 0.6498132183908045

=== Iteration 85000 ===
np.mean(results), mse, len(results) =  0.6386274509803922 tf.Tensor(3546.7228032504445, shape=(), dtype=float64) 51
slice = key, score = 0.6386274509803922
np.mean(results), mse, len(results) =  0.6730566507574186 tf.Tensor(3489.4254804995867, shape=(), dtype=float64) 4819
slice = train, score = 0.6730566507574186
np.mean(results), mse, len(results) =  0.6611973180076628 tf.Tensor(3486.565577677944, shape=(), dtype=float64) 2088
slice = test, score = 0.6611973180076628

=== Iteration 86000 ===
np.mean(results), mse, len(results) =  0.6284313725490196 tf.Tensor(3644.5963414025086, shape=(), dtype=float64) 51
slice = key, score = 0.6284313725490196
np

np.mean(results), mse, len(results) =  0.6732807636439095 tf.Tensor(3875.30276296401, shape=(), dtype=float64) 4819
np.mean(results), mse, len(results) =  0.6618055555555556 tf.Tensor(3873.109539148968, shape=(), dtype=float64) 2088
0.6732807636439095 0.6618055555555556


In [26]:
GE_QE_s = [
    "GE_QE_AnnCURxCoItem_RecSysLT_Round10_6565.npz",
    "GE_QE_AnnCURxKMeans_RecSysLT_Round10_613.npz",
    "GE_QE_AnnCURxRandom_RecSysLT_Round10_5842.npz",
    "GGE_QE_AnnCURxTop_RecSysLT_Round10_6695.npz"
]

In [29]:
res = dict()
for GE_QE_i in GE_QE_s:
    loaded = np.load('./' + GE_QE_i)
    res["GE_QE_i"] = do_AXN(ctx, loaded, get_Rp_basic)

100%|█████████████████████████████████████████| 104/104 [00:26<00:00,  3.88it/s]


K = 25, L = 0.0, TS = 100, rndFirst = False np.mean(results) = 0.8504807692307692


100%|█████████████████████████████████████████| 104/104 [00:30<00:00,  3.43it/s]


K = 25, L = 0.1, TS = 100, rndFirst = False np.mean(results) = 0.8435576923076922


100%|█████████████████████████████████████████| 104/104 [00:25<00:00,  4.12it/s]


...

100%|█████████████████████████████████████████| 104/104 [00:29<00:00,  3.58it/s]


K = 25, L = 0.9, TS = 100, rndFirst = False np.mean(results) = 0.7331730769230771


100%|█████████████████████████████████████████| 104/104 [00:27<00:00,  3.73it/s]


K = 25, L = 1.0, TS = 100, rndFirst = False np.mean(results) = 0.720576923076923


FileNotFoundError: [Errno 2] No such file or directory: './GGE_QE_AnnCURxTop_RecSysLT_Round10_6695.npz'

In [30]:
GE_QE_s[-1] = "GE_QE_AnnCURxTop_RecSysLT_Round10_6695.npz"

In [31]:
res = dict()
for GE_QE_i in GE_QE_s[-1:]:
    loaded = np.load('./' + GE_QE_i)
    res["GE_QE_i"] = do_AXN(ctx, loaded, get_Rp_basic)

100%|█████████████████████████████████████████| 104/104 [00:30<00:00,  3.42it/s]


K = 25, L = 0.0, TS = 100, rndFirst = False np.mean(results) = 0.8509615384615384


100%|█████████████████████████████████████████| 104/104 [00:27<00:00,  3.72it/s]


K = 25, L = 0.1, TS = 100, rndFirst = False np.mean(results) = 0.8435576923076924


100%|█████████████████████████████████████████| 104/104 [00:24<00:00,  4.32it/s]


K = 25, L = 0.2, TS = 100, rndFirst = False np.mean(results) = 0.8430769230769232


100%|█████████████████████████████████████████| 104/104 [00:31<00:00,  3.34it/s]


K = 25, L = 0.30000000000000004, TS = 100, rndFirst = False np.mean(results) = 0.8350000000000001


 95%|███████████████████████████████████████▉  | 99/104 [00:28<00:01,  3.50it/s]


KeyboardInterrupt: 

In [57]:
res

{'GE_QE_AnnCURxCoItem_yugioh_Round10_5617.npz': 'K = 25, L = 0.0, rndFirst = False np.mean(results) = 0.7226',
 'GE_QE_AnnCURxKMeans_yugioh_Round10_5017.npz': 'K = 25, L = 0.0, rndFirst = False np.mean(results) = 0.6807999999999998',
 'GE_QE_AnnCURxTop_yugioh_Round10_2429.npz': 'K = 25, L = 0.3, rndFirst = False np.mean(results) = 0.38',
 'GE_QE_AnnCURxRandom_yugioh_Round10_4721.npz': 'K = 25, L = 0.0, rndFirst = False np.mean(results) = 0.6146'}

## RecSys PairWise

In [28]:
L = 7000
N = 1000
DA = 50


ctx = load(L, raw_path = "//dev/null/stand/log.local.txt",
           path="log.local.bin.old", seed=17, det_attempts=DA, key_size=50)
print("LOADED")

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  1.4692355991364834e-44
1.4692355991364834e-44
51 4694 2034
LOADED


In [29]:
do(ctx, "RecSysLT_Key50_")

/tmp/ipykernel_965688/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/50 [00:00<?, ?it/s]

np.mean(results), mse, len(results) =  0.6523753728163613 0.4232415359177667 4694
np.mean(results), mse, len(results) =  0.6526745329400198 0.5435502971951501 2034
./GE_QE_AnnCURxCoItem_RecSysLT_Key50__6527.npz


/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/base.py:1151: ConvergenceWarning: Number of distinct clusters (42) found smaller than n_clusters (50). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


np.mean(results), mse, len(results) =  0.7216744780570942 1.1133082210796557 4694
np.mean(results), mse, len(results) =  0.7174188790560472 1.0786282255946333 2034
./GE_QE_AnnCURxKMeans_RecSysLT_Key50__7174.npz
np.mean(results), mse, len(results) =  0.7695398380911803 0.5488957119002489 4694
np.mean(results), mse, len(results) =  0.7671976401179941 0.5850987156264035 2034
./GE_QE_AnnCURxTop_RecSysLT_Key50__7672.npz
np.mean(results), mse, len(results) =  0.6617895185342991 2.169721236214458 4694
np.mean(results), mse, len(results) =  0.6611553588987217 1.918897574185812 2034
./GE_QE_AnnCURxRandom_RecSysLT_Key50__6612.npz


In [31]:
do_RBE(ctx, "RecSysLT_Key50_")

/tmp/ipykernel_965688/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

self.embed_users['train'].shape =  (4694, 100)
self.embed_games.shape =  (16514, 51)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (51, 16514)
self.core_users_embs.shape =  (51, 100)
self.ge_users.shape =  (4694, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.19137254901960785 tf.Tensor(41.028397650803306, shape=(), dtype=float64) 51
slice = key, score = 0.19137254901960785
np.mean(results), mse, len(results) =  0.281011930123562 tf.Tensor(27.50619103581194, shape=(), dtype=float64) 4694
slice = train, score = 0.281011930123562
np.mean(results), mse, len(results) =  0.21592428711897738 tf.Tensor(28.28270321926398, shape=(), dtype=float64) 2034
slice = test, score = 0.21592428711897738
np.mean(results), mse, len(results) =  0.21592428711897738 tf.Tensor(28.28270321926398, shape=(), dtype=float64) 2034
Saving ./GE_QE_RBExCoItem_RecSysLT_Key50__0.2159.npz ...

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.59490196078

np.mean(results), mse, len(results) =  0.6883677482792526 tf.Tensor(3237.21066757381, shape=(), dtype=float64) 2034
slice = test, score = 0.6883677482792526

=== Iteration 13000 ===
np.mean(results), mse, len(results) =  0.6588235294117647 tf.Tensor(8184.4038302252775, shape=(), dtype=float64) 51
slice = key, score = 0.6588235294117647
np.mean(results), mse, len(results) =  0.6995654026416702 tf.Tensor(3045.1411383003433, shape=(), dtype=float64) 4694
slice = train, score = 0.6995654026416702
np.mean(results), mse, len(results) =  0.6915683382497543 tf.Tensor(3190.306802601881, shape=(), dtype=float64) 2034
slice = test, score = 0.6915683382497543

=== Iteration 14000 ===
np.mean(results), mse, len(results) =  0.6641176470588236 tf.Tensor(9596.730183152255, shape=(), dtype=float64) 51
slice = key, score = 0.6641176470588236
np.mean(results), mse, len(results) =  0.6995100127822752 tf.Tensor(3723.5105549378623, shape=(), dtype=float64) 4694
slice = train, score = 0.6995100127822752
np.m


=== Iteration 28000 ===
np.mean(results), mse, len(results) =  0.6776470588235295 tf.Tensor(22044.36974105708, shape=(), dtype=float64) 51
slice = key, score = 0.6776470588235295
np.mean(results), mse, len(results) =  0.7152577758841074 tf.Tensor(8030.714460706056, shape=(), dtype=float64) 4694
slice = train, score = 0.7152577758841074
np.mean(results), mse, len(results) =  0.7062881022615536 tf.Tensor(8481.855339965894, shape=(), dtype=float64) 2034
slice = test, score = 0.7062881022615536
np.mean(results), mse, len(results) =  0.7062881022615536 tf.Tensor(8481.855339965894, shape=(), dtype=float64) 2034
Saving ./GE_QE_RBExCoItem_RecSysLT_Key50__0.7063.npz ...

=== Iteration 29000 ===
np.mean(results), mse, len(results) =  0.6652941176470587 tf.Tensor(27020.160672314665, shape=(), dtype=float64) 51
slice = key, score = 0.6652941176470587
np.mean(results), mse, len(results) =  0.7135769066893907 tf.Tensor(10350.084978246532, shape=(), dtype=float64) 4694
slice = train, score = 0.71357

...

np.mean(results), mse, len(results) =  0.7463274336283187 tf.Tensor(8587.21558888933, shape=(), dtype=float64) 2034
Saving ./GE_QE_RecSysLT_Key50__RBExRandom_0.7463.npz ...

=== Iteration 36000 ===
np.mean(results), mse, len(results) =  0.7152941176470587 tf.Tensor(11263.667821396259, shape=(), dtype=float64) 51
slice = key, score = 0.7152941176470587
np.mean(results), mse, len(results) =  0.7473157221985514 tf.Tensor(7092.332086147643, shape=(), dtype=float64) 4694
slice = train, score = 0.7473157221985514
np.mean(results), mse, len(results) =  0.7442084562438545 tf.Tensor(7383.794041328882, shape=(), dtype=float64) 2034
slice = test, score = 0.7442084562438545

=== Iteration 37000 ===
np.mean(results), mse, len(results) =  0.6962745098039217 tf.Tensor(15750.27214473973, shape=(), dtype=float64) 51
slice = key, score = 0.6962745098039217
np.mean(results), mse, len(results) =  0.7464294844482318 tf.Tensor(7922.542910809092, shape=(), dtype=float64) 4694
slice = train, score = 0.7464294

np.mean(results), mse, len(results) =  0.7507620452310718 tf.Tensor(12614.966197193493, shape=(), dtype=float64) 2034
slice = test, score = 0.7507620452310718
np.mean(results), mse, len(results) =  0.7507620452310718 tf.Tensor(12614.966197193493, shape=(), dtype=float64) 2034
Saving ./GE_QE_RecSysLT_Key50__RBExRandom_0.7508.npz ...

=== Iteration 51000 ===
np.mean(results), mse, len(results) =  0.7029411764705883 tf.Tensor(25605.978333231367, shape=(), dtype=float64) 51
slice = key, score = 0.7029411764705883
np.mean(results), mse, len(results) =  0.7524989348103962 tf.Tensor(12483.95065980334, shape=(), dtype=float64) 4694
slice = train, score = 0.7524989348103962
np.mean(results), mse, len(results) =  0.7444444444444444 tf.Tensor(13165.614677592785, shape=(), dtype=float64) 2034
slice = test, score = 0.7444444444444444

=== Iteration 52000 ===
np.mean(results), mse, len(results) =  0.7158823529411765 tf.Tensor(23381.48794371453, shape=(), dtype=float64) 51
slice = key, score = 0.7158


=== Iteration 66000 ===
np.mean(results), mse, len(results) =  0.7123529411764704 tf.Tensor(33911.78204414843, shape=(), dtype=float64) 51
slice = key, score = 0.7123529411764704
np.mean(results), mse, len(results) =  0.7557584149978697 tf.Tensor(18329.307105029435, shape=(), dtype=float64) 4694
slice = train, score = 0.7557584149978697
np.mean(results), mse, len(results) =  0.7487905604719763 tf.Tensor(19359.129774225203, shape=(), dtype=float64) 2034
slice = test, score = 0.7487905604719763

=== Iteration 67000 ===
np.mean(results), mse, len(results) =  0.7241176470588235 tf.Tensor(35032.613093611384, shape=(), dtype=float64) 51
slice = key, score = 0.7241176470588235
np.mean(results), mse, len(results) =  0.7601065189603748 tf.Tensor(19207.199816707973, shape=(), dtype=float64) 4694
slice = train, score = 0.7601065189603748
np.mean(results), mse, len(results) =  0.7552949852507375 tf.Tensor(20072.140416707414, shape=(), dtype=float64) 2034
slice = test, score = 0.7552949852507375
n

np.mean(results), mse, len(results) =  0.7517699115044247 tf.Tensor(27710.473086261063, shape=(), dtype=float64) 2034
slice = test, score = 0.7517699115044247

=== Iteration 82000 ===
np.mean(results), mse, len(results) =  0.7137254901960784 tf.Tensor(54759.05111154787, shape=(), dtype=float64) 51
slice = key, score = 0.7137254901960784
np.mean(results), mse, len(results) =  0.7534128674904133 tf.Tensor(27095.75655108125, shape=(), dtype=float64) 4694
slice = train, score = 0.7534128674904133
np.mean(results), mse, len(results) =  0.7458308751229105 tf.Tensor(28745.105617816756, shape=(), dtype=float64) 2034
slice = test, score = 0.7458308751229105

=== Iteration 83000 ===
np.mean(results), mse, len(results) =  0.7282352941176471 tf.Tensor(48919.34463545443, shape=(), dtype=float64) 51
slice = key, score = 0.7282352941176471
np.mean(results), mse, len(results) =  0.7622049424797613 tf.Tensor(26378.488780019634, shape=(), dtype=float64) 4694
slice = train, score = 0.7622049424797613
np.


=== Iteration 98000 ===
np.mean(results), mse, len(results) =  0.7170588235294117 tf.Tensor(70231.3952952414, shape=(), dtype=float64) 51
slice = key, score = 0.7170588235294117
np.mean(results), mse, len(results) =  0.7592458457605454 tf.Tensor(36390.9648993698, shape=(), dtype=float64) 4694
slice = train, score = 0.7592458457605454
np.mean(results), mse, len(results) =  0.7510766961651918 tf.Tensor(38083.529275474975, shape=(), dtype=float64) 2034
slice = test, score = 0.7510766961651918

=== Iteration 99000 ===
np.mean(results), mse, len(results) =  0.703529411764706 tf.Tensor(76204.88118972903, shape=(), dtype=float64) 51
slice = key, score = 0.703529411764706
np.mean(results), mse, len(results) =  0.7614806135492117 tf.Tensor(35584.69502805213, shape=(), dtype=float64) 4694
slice = train, score = 0.7614806135492117
np.mean(results), mse, len(results) =  0.7551081612586037 tf.Tensor(37548.16732348121, shape=(), dtype=float64) 2034
slice = test, score = 0.7551081612586037
last loss

In [34]:
!ls | grep "GE.*RecSys_"

GE_QE_AnnCURxCoItem_RecSys_Round10_7197.npz
GE_QE_AnnCURxKMeans_RecSys_Round10_7156.npz
GE_QE_AnnCURxRandom_RecSys_Round10_6743.npz
GE_QE_AnnCURxTop_RecSys_Round10_7623.npz


In [35]:
GE_QE_s = [
    "GE_QE_AnnCURxCoItem_RecSys_Round10_7197.npz",
    "GE_QE_AnnCURxKMeans_RecSys_Round10_7156.npz",
    "GE_QE_AnnCURxRandom_RecSys_Round10_6743.npz",
    "GE_QE_AnnCURxTop_RecSys_Round10_7623.npz"
]

In [40]:
res = dict()
for GE_QE_i in GE_QE_s:
    loaded = np.load('./' + GE_QE_i)
    res[GE_QE_i] = do_AXN(ctx, loaded, get_Rp_basic, Ls = np.linspace(0, 0.3, 4))

100%|█████████████████████████████████████████| 101/101 [00:14<00:00,  6.84it/s]


K = 25, L = 0.0, TS = 100, rndFirst = False np.mean(results) = 0.8829702970297029


100%|█████████████████████████████████████████| 101/101 [00:13<00:00,  7.34it/s]


K = 25, L = 0.09999999999999999, TS = 100, rndFirst = False np.mean(results) = 0.7820792079207922


100%|█████████████████████████████████████████| 101/101 [00:15<00:00,  6.66it/s]


...

K = 25, L = 0.09999999999999999, TS = 100, rndFirst = False np.mean(results) = 0.858019801980198


100%|█████████████████████████████████████████| 101/101 [00:08<00:00, 12.04it/s]


K = 25, L = 0.19999999999999998, TS = 100, rndFirst = False np.mean(results) = 0.8446534653465346


100%|█████████████████████████████████████████| 101/101 [00:12<00:00,  7.85it/s]

K = 25, L = 0.3, TS = 100, rndFirst = False np.mean(results) = 0.8324752475247525


In [41]:
res

{'GE_QE_AnnCURxCoItem_RecSys_Round10_7197.npz': 'K = 25, L = 0.0, rndFirst = False np.mean(results) = 0.8829702970297029',
 'GE_QE_AnnCURxKMeans_RecSys_Round10_7156.npz': 'K = 25, L = 0.0, rndFirst = False np.mean(results) = 0.9028712871287127',
 'GE_QE_AnnCURxRandom_RecSys_Round10_6743.npz': 'K = 25, L = 0.0, rndFirst = False np.mean(results) = 0.8820792079207923',
 'GE_QE_AnnCURxTop_RecSys_Round10_7623.npz': 'K = 25, L = 0.0, rndFirst = False np.mean(results) = 0.9113861386138613'}

In [ ]:
AnnCURxTop_RecSys : 0.9113861386138613 -> 0.9113861386138613
AnnCURxCoItem_RecSys : 0.8829702970297029 -> 0.8829702970297029
AnnCURxKMeans_RecSys : 0.9028712871287127 -> 0.9028712871287127
AnnCURxRandom_RecSys : 0.8820792079207923 -> 0.8820792079207923

# ZESHEL

In [39]:
DATASETS = ["yugioh", "pro_wrestling", "star_trek", "doctor_who", "military"]

In [40]:
del ctx

In [41]:
gc.collect()

13

In [42]:
for dataset in DATASETS:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100,
        key_size=50
    )

    print("LOADED")
    
    do(ctx, f"{dataset}_Key50")
    
    del ctx
    gc.collect()




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (0, 1070604160.0 -> 4473067008.0)
updated det (3, 4473067008.0 -> 6631089664.0)
updated det (4, 6631089664.0 -> 69104549888.0)
updated det (7, 69104549888.0 -> 1087518670848.0)
updated det (8, 1087518670848.0 -> 6136270422016.0)
updated det (23, 6136270422016.0 -> 92915267272704.0)
updated det (64, 92915267272704.0 -> 2.671569730849997e+16)


/tmp/ipykernel_965688/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/50 [00:00<?, ?it/s]

np.mean(results), mse, len(results) =  0.4890909090909091 0.1677604 2310
np.mean(results), mse, len(results) =  0.46759131293188555 0.17839523 1013
./GE_QE_AnnCURxCoItem_yugioh_Key50_4676.npz


/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


...

/tmp/ipykernel_965688/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/50 [00:00<?, ?it/s]

np.mean(results), mse, len(results) =  0.32493554327808466 0.007821261 1629
np.mean(results), mse, len(results) =  0.29459722222222223 0.011325857 720
./GE_QE_AnnCURxCoItem_military_Key50_2946.npz


/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


np.mean(results), mse, len(results) =  0.33092081031307546 0.0076574874 1629
np.mean(results), mse, len(results) =  0.28447222222222224 0.009860543 720
./GE_QE_AnnCURxKMeans_military_Key50_2845.npz
np.mean(results), mse, len(results) =  0.19683855125844074 0.01133361 1629
np.mean(results), mse, len(results) =  0.17355555555555557 0.0139393 720
./GE_QE_AnnCURxTop_military_Key50_1736.npz
np.mean(results), mse, len(results) =  0.24073050951503988 0.009288687 1629
np.mean(results), mse, len(results) =  0.22306944444444446 0.011516683 720
./GE_QE_AnnCURxRandom_military_Key50_2231.npz


In [ ]:
for dataset in DATASETS:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100,
        key_size=50
    )

    print("LOADED")
    
    do_RBE(ctx, f"{dataset}_Key50")
    
    del ctx
    gc.collect()




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (0, 1070604160.0 -> 4473067008.0)
updated det (3, 4473067008.0 -> 6631089664.0)
updated det (4, 6631089664.0 -> 69104549888.0)
updated det (7, 69104549888.0 -> 1087518670848.0)
updated det (8, 1087518670848.0 -> 6136270422016.0)
updated det (23, 6136270422016.0 -> 92915267272704.0)
updated det (64, 92915267272704.0 -> 2.671569730849997e+16)


/tmp/ipykernel_965688/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

self.embed_users['train'].shape =  (2310, 100)
self.embed_games.shape =  (10031, 50)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (50, 10031)
self.core_users_embs.shape =  (50, 100)
self.ge_users.shape =  (2310, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0384 tf.Tensor(39.533638, shape=(), dtype=float32) 50
slice = key, score = 0.0384
np.mean(results), mse, len(results) =  0.04012121212121213 tf.Tensor(42.28699, shape=(), dtype=float32) 2310
slice = train, score = 0.04012121212121213
np.mean(results), mse, len(results) =  0.03675222112537019 tf.Tensor(42.036404, shape=(), dtype=float32) 1013
slice = test, score = 0.03675222112537019
np.mean(results), mse, len(results) =  0.03675222112537019 tf.Tensor(42.036404, shape=(), dtype=float32) 1013
Saving ./GE_QE_RBExCoItem_yugioh_Key50_0.0368.npz ...

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.545 tf.Tensor(85.60582, shape=(), dtype=float32) 50
slice = key, score


=== Iteration 13000 ===
np.mean(results), mse, len(results) =  0.6036 tf.Tensor(989.88354, shape=(), dtype=float32) 50
slice = key, score = 0.6036
np.mean(results), mse, len(results) =  0.6279480519480519 tf.Tensor(972.1143, shape=(), dtype=float32) 2310
slice = train, score = 0.6279480519480519
np.mean(results), mse, len(results) =  0.5737907206317868 tf.Tensor(970.0473, shape=(), dtype=float32) 1013
slice = test, score = 0.5737907206317868
np.mean(results), mse, len(results) =  0.5737907206317868 tf.Tensor(970.0473, shape=(), dtype=float32) 1013
Saving ./GE_QE_RBExCoItem_yugioh_Key50_0.5738.npz ...

=== Iteration 14000 ===
np.mean(results), mse, len(results) =  0.6053999999999999 tf.Tensor(1101.2938, shape=(), dtype=float32) 50
slice = key, score = 0.6053999999999999
np.mean(results), mse, len(results) =  0.6293766233766235 tf.Tensor(1085.7264, shape=(), dtype=float32) 2310
slice = train, score = 0.6293766233766235
np.mean(results), mse, len(results) =  0.5749555774925963 tf.Tensor(

...

np.mean(results), mse, len(results) =  0.15306501547987617 tf.Tensor(4468.3955, shape=(), dtype=float32) 2907
slice = train, score = 0.15306501547987617
np.mean(results), mse, len(results) =  0.114822695035461 tf.Tensor(4533.479, shape=(), dtype=float32) 1269
slice = test, score = 0.114822695035461
np.mean(results), mse, len(results) =  0.114822695035461 tf.Tensor(4533.479, shape=(), dtype=float32) 1269
Saving ./GE_QE_star_trek_Key50_RBExRandom_0.1148.npz ...

=== Iteration 28000 ===
np.mean(results), mse, len(results) =  0.11120000000000001 tf.Tensor(4827.441, shape=(), dtype=float32) 50
slice = key, score = 0.11120000000000001
np.mean(results), mse, len(results) =  0.154234606123151 tf.Tensor(4807.6924, shape=(), dtype=float32) 2907
slice = train, score = 0.154234606123151
np.mean(results), mse, len(results) =  0.11490149724192278 tf.Tensor(4881.9043, shape=(), dtype=float32) 1269
slice = test, score = 0.11490149724192278
np.mean(results), mse, len(results) =  0.11490149724192278 tf.

np.mean(results), mse, len(results) =  0.11454688731284476 tf.Tensor(9346.11, shape=(), dtype=float32) 1269
slice = test, score = 0.11454688731284476

=== Iteration 42000 ===
np.mean(results), mse, len(results) =  0.12179999999999999 tf.Tensor(9661.559, shape=(), dtype=float32) 50
slice = key, score = 0.12179999999999999
np.mean(results), mse, len(results) =  0.16198486412108704 tf.Tensor(9687.159, shape=(), dtype=float32) 2907
slice = train, score = 0.16198486412108704
np.mean(results), mse, len(results) =  0.11652482269503546 tf.Tensor(9597.037, shape=(), dtype=float32) 1269
slice = test, score = 0.11652482269503546
np.mean(results), mse, len(results) =  0.11652482269503546 tf.Tensor(9597.037, shape=(), dtype=float32) 1269
Saving ./GE_QE_star_trek_Key50_RBExRandom_0.1165.npz ...

=== Iteration 43000 ===
np.mean(results), mse, len(results) =  0.12040000000000001 tf.Tensor(10181.465, shape=(), dtype=float32) 50
slice = key, score = 0.12040000000000001
np.mean(results), mse, len(results

np.mean(results), mse, len(results) =  0.11666666666666668 tf.Tensor(18230.404, shape=(), dtype=float32) 1269
slice = test, score = 0.11666666666666668
np.mean(results), mse, len(results) =  0.11666666666666668 tf.Tensor(18230.404, shape=(), dtype=float32) 1269
Saving ./GE_QE_star_trek_Key50_RBExRandom_0.1167.npz ...

=== Iteration 58000 ===
np.mean(results), mse, len(results) =  0.12180000000000002 tf.Tensor(19056.37, shape=(), dtype=float32) 50
slice = key, score = 0.12180000000000002
np.mean(results), mse, len(results) =  0.16674922600619196 tf.Tensor(19356.865, shape=(), dtype=float32) 2907
slice = train, score = 0.16674922600619196
np.mean(results), mse, len(results) =  0.11761229314420804 tf.Tensor(19017.39, shape=(), dtype=float32) 1269
slice = test, score = 0.11761229314420804

=== Iteration 59000 ===
np.mean(results), mse, len(results) =  0.1206 tf.Tensor(19628.139, shape=(), dtype=float32) 50
slice = key, score = 0.1206
np.mean(results), mse, len(results) =  0.165909872721018

np.mean(results), mse, len(results) =  0.11847911741528762 tf.Tensor(30383.686, shape=(), dtype=float32) 1269
slice = test, score = 0.11847911741528762

=== Iteration 74000 ===
np.mean(results), mse, len(results) =  0.12500000000000003 tf.Tensor(31126.709, shape=(), dtype=float32) 50
slice = key, score = 0.12500000000000003
np.mean(results), mse, len(results) =  0.17244582043343654 tf.Tensor(31643.607, shape=(), dtype=float32) 2907
slice = train, score = 0.17244582043343654
np.mean(results), mse, len(results) =  0.11823483057525612 tf.Tensor(31138.936, shape=(), dtype=float32) 1269
slice = test, score = 0.11823483057525612
np.mean(results), mse, len(results) =  0.11823483057525612 tf.Tensor(31138.936, shape=(), dtype=float32) 1269
Saving ./GE_QE_star_trek_Key50_RBExRandom_0.1182.npz ...

=== Iteration 75000 ===
np.mean(results), mse, len(results) =  0.12580000000000002 tf.Tensor(31973.4, shape=(), dtype=float32) 50
slice = key, score = 0.12580000000000002
np.mean(results), mse, len(res

np.mean(results), mse, len(results) =  0.11821907013396375 tf.Tensor(43749.414, shape=(), dtype=float32) 1269
slice = test, score = 0.11821907013396375

=== Iteration 91000 ===
np.mean(results), mse, len(results) =  0.12520000000000003 tf.Tensor(45357.855, shape=(), dtype=float32) 50
slice = key, score = 0.12520000000000003
np.mean(results), mse, len(results) =  0.1741967664258686 tf.Tensor(46085.965, shape=(), dtype=float32) 2907
slice = train, score = 0.1741967664258686
np.mean(results), mse, len(results) =  0.11745468873128448 tf.Tensor(45458.652, shape=(), dtype=float32) 1269
slice = test, score = 0.11745468873128448

=== Iteration 92000 ===
np.mean(results), mse, len(results) =  0.1294 tf.Tensor(46331.16, shape=(), dtype=float32) 50
slice = key, score = 0.1294
np.mean(results), mse, len(results) =  0.17486756105951154 tf.Tensor(47071.203, shape=(), dtype=float32) 2907
slice = train, score = 0.17486756105951154
np.mean(results), mse, len(results) =  0.11815602836879432 tf.Tensor(46

In [ ]:
ress = dict()
for dataset in DATASETS[:1]:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100
    )

    print("LOADED")
    
    
    GE_QE_s = [f for f in os.listdir("./") if "GE_QE_Ann" in f and "yugioh" in f]
    
    ress[dataset] = res = dict()
    
    for GE_QE_i in GE_QE_s:
        print("GE_QE_i = ", GE_QE_i)
        loaded = np.load('./' + GE_QE_i)
        ress[dataset][GE_QE_i] = do_AXN(ctx, loaded, get_Rp_basic, Ls = np.linspace(0, 0.3, 4))
    
    del ctx
    gc.collect()

In [60]:
ress = dict()
for dataset in DATASETS[:1]:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100
    )

    print("LOADED")
    
    
    GE_QE_s = [f for f in os.listdir("./") if "GE_QE_Ann" in f and "yugioh" in f]
    
    ress[dataset] = res = dict()
    
    for GE_QE_i in GE_QE_s:
        print("GE_QE_i = ", GE_QE_i)
        loaded = np.load('./' + GE_QE_i)
        ress[dataset][GE_QE_i] = do_AXN(ctx, loaded, get_Rp_basic, Ls = np.linspace(0, 0.3, 4))
    
    del ctx
    gc.collect()




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013
LOADED
GE_QE_i =  GE_QE_AnnCURxCoItem_yugioh_Round10_5617.npz


100%|███████████████████████████████████████████| 50/50 [00:02<00:00, 17.00it/s]


K = 25, L = 0.0, TS = 100, rndFirst = False np.mean(results) = 0.7226 best = 0.7226


100%|███████████████████████████████████████████| 50/50 [00:04<00:00, 11.45it/s]


K = 25, L = 0.09999999999999999, TS = 100, rndFirst = False np.mean(results) = 0.588 best = 0.7226


...

K = 25, L = 0.09999999999999999, TS = 100, rndFirst = False np.mean(results) = 0.5218 best = 0.6146


100%|███████████████████████████████████████████| 50/50 [00:04<00:00, 10.70it/s]


K = 25, L = 0.19999999999999998, TS = 100, rndFirst = False np.mean(results) = 0.5067999999999999 best = 0.6146


100%|███████████████████████████████████████████| 50/50 [00:04<00:00, 11.97it/s]


K = 25, L = 0.3, TS = 100, rndFirst = False np.mean(results) = 0.5132000000000001 best = 0.6146


In [61]:
ress

{'yugioh': {'GE_QE_AnnCURxCoItem_yugioh_Round10_5617.npz': 'K = 25, L = 0.0, rndFirst = False np.mean(results) = 0.7226',
  'GE_QE_AnnCURxKMeans_yugioh_Round10_5017.npz': 'K = 25, L = 0.0, rndFirst = False np.mean(results) = 0.6807999999999998',
  'GE_QE_AnnCURxTop_yugioh_Round10_2429.npz': 'K = 25, L = 0.3, rndFirst = False np.mean(results) = 0.38',
  'GE_QE_AnnCURxRandom_yugioh_Round10_4721.npz': 'K = 25, L = 0.0, rndFirst = False np.mean(results) = 0.6146'}}

In [65]:
ress = dict()
for dataset in DATASETS:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100
    )

    print("LOADED")
    
    
    GE_QE_s = [
        f for f in os.listdir("./")
        if ("GE_QE_Ann" in f) and (dataset in f)
    ]
    
    ress[dataset] = res = dict()
    
    for GE_QE_i in GE_QE_s:
        print("GE_QE_i = ", GE_QE_i)
        loaded = np.load('./' + GE_QE_i)
        ress[dataset][GE_QE_i] = do_AXN(ctx, loaded, get_Rp_basic)
    
    del ctx
    gc.collect()




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013
LOADED
GE_QE_i =  GE_QE_AnnCURxCoItem_yugioh_Round10_5617.npz


100%|███████████████████████████████████████████| 50/50 [00:03<00:00, 12.69it/s]


K = 25, L = 0.0, TS = 100, rndFirst = False np.mean(results) = 0.7226 best = 0.7226


100%|███████████████████████████████████████████| 50/50 [00:06<00:00,  7.16it/s]


K = 25, L = 0.1, TS = 100, rndFirst = False np.mean(results) = 0.588 best = 0.7226


...

K = 25, L = 0.8, TS = 100, rndFirst = False np.mean(results) = 0.22055555555555556 best = 0.3350000000000001


100%|███████████████████████████████████████████| 36/36 [00:11<00:00,  3.14it/s]


K = 25, L = 0.9, TS = 100, rndFirst = False np.mean(results) = 0.205 best = 0.3350000000000001


100%|███████████████████████████████████████████| 36/36 [00:13<00:00,  2.64it/s]

K = 25, L = 1.0, TS = 100, rndFirst = False np.mean(results) = 0.19944444444444442 best = 0.3350000000000001


In [66]:
ress

{'yugioh': {'GE_QE_AnnCURxCoItem_yugioh_Round10_5617.npz': 'K = 25, L = 0.0, rndFirst = False np.mean(results) = 0.7226',
  'GE_QE_AnnCURxKMeans_yugioh_Round10_5017.npz': 'K = 25, L = 0.0, rndFirst = False np.mean(results) = 0.6807999999999998',
  'GE_QE_AnnCURxTop_yugioh_Round10_2429.npz': 'K = 25, L = 0.4, rndFirst = False np.mean(results) = 0.3874',
  'GE_QE_AnnCURxRandom_yugioh_Round10_4721.npz': 'K = 25, L = 0.0, rndFirst = False np.mean(results) = 0.6146'},
 'pro_wrestling': {'GE_QE_AnnCURxKMeans_pro_wrestling_Round10_4834.npz': 'K = 25, L = 0.0, rndFirst = False np.mean(results) = 0.616',
  'GE_QE_AnnCURxRandom_pro_wrestling_Round10_4279.npz': 'K = 25, L = 0.0, rndFirst = False np.mean(results) = 0.5724999999999999',
  'GE_QE_AnnCURxTop_pro_wrestling_Round10_3002.npz': 'K = 25, L = 0.0, rndFirst = False np.mean(results) = 0.43499999999999994',
  'GE_QE_AnnCURxCoItem_pro_wrestling_Round10_5119.npz': 'K = 25, L = 0.0, rndFirst = False np.mean(results) = 0.6635'},
 'star_trek': {'G

In [68]:
ress = dict()
for dataset in DATASETS:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100
    )

    print("LOADED")
    
    
    GE_QE_s = [
        f for f in os.listdir("./")
        if ("GE_QE_Ann" in f) and (dataset in f)
    ]
    
    ress[dataset] = res = dict()
    
    for GE_QE_i in GE_QE_s:
        print("GE_QE_i = ", GE_QE_i)
        loaded = np.load('./' + GE_QE_i)
        ress[dataset][GE_QE_i] = do_AXN(ctx, loaded, get_Rp_basic, STRIP=0.1, Ks=[10, 25, 50])
    
    del ctx
    gc.collect()




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013
LOADED
GE_QE_i =  GE_QE_AnnCURxCoItem_yugioh_Round10_5617.npz


100%|█████████████████████████████████████████| 101/101 [00:25<00:00,  3.93it/s]


K = 10, L = 0.0, TS = 100, rndFirst = False np.mean(results) = 0.7172277227722772 best = 0.7172277227722772


100%|█████████████████████████████████████████| 101/101 [00:25<00:00,  3.95it/s]


K = 10, L = 0.1, TS = 100, rndFirst = False np.mean(results) = 0.597920792079208 best = 0.7172277227722772


...

100%|███████████████████████████████████████████| 72/72 [00:35<00:00,  2.02it/s]


K = 10, L = 0.30000000000000004, TS = 100, rndFirst = False np.mean(results) = 0.33444444444444443 best = 0.42541666666666667


100%|███████████████████████████████████████████| 72/72 [00:34<00:00,  2.09it/s]


K = 10, L = 0.4, TS = 100, rndFirst = False np.mean(results) = 0.3209722222222222 best = 0.42541666666666667


 50%|█████████████████████▌                     | 36/72 [00:17<00:19,  1.82it/s]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [69]:
ress

{'yugioh': {'GE_QE_AnnCURxCoItem_yugioh_Round10_5617.npz': 'K = 10, L = 0.0, rndFirst = False np.mean(results) = 0.7172277227722772',
  'GE_QE_AnnCURxKMeans_yugioh_Round10_5017.npz': 'K = 10, L = 0.0, rndFirst = False np.mean(results) = 0.6592079207920792',
  'GE_QE_AnnCURxTop_yugioh_Round10_2429.npz': 'K = 10, L = 0.8, rndFirst = False np.mean(results) = 0.4597029702970298',
  'GE_QE_AnnCURxRandom_yugioh_Round10_4721.npz': 'K = 10, L = 0.0, rndFirst = False np.mean(results) = 0.6263366336633663'},
 'pro_wrestling': {'GE_QE_AnnCURxKMeans_pro_wrestling_Round10_4834.npz': 'K = 10, L = 0.0, rndFirst = False np.mean(results) = 0.6531707317073171',
  'GE_QE_AnnCURxRandom_pro_wrestling_Round10_4279.npz': 'K = 10, L = 0.0, rndFirst = False np.mean(results) = 0.5895121951219512',
  'GE_QE_AnnCURxTop_pro_wrestling_Round10_3002.npz': 'K = 10, L = 0.0, rndFirst = False np.mean(results) = 0.44536585365853665',
  'GE_QE_AnnCURxCoItem_pro_wrestling_Round10_5119.npz': 'K = 10, L = 0.0, rndFirst = Fal

In [78]:
import json

for key in ress.keys():
    print()
    print(f"================== {key} ==================")
    print(json.dumps(ress[key], indent=""))


================== yugioh ==================
{
"GE_QE_AnnCURxCoItem_yugioh_Round10_5617.npz": "K = 10, L = 0.0, rndFirst = False np.mean(results) = 0.7172277227722772",
"GE_QE_AnnCURxKMeans_yugioh_Round10_5017.npz": "K = 10, L = 0.0, rndFirst = False np.mean(results) = 0.6592079207920792",
"GE_QE_AnnCURxTop_yugioh_Round10_2429.npz": "K = 10, L = 0.8, rndFirst = False np.mean(results) = 0.4597029702970298",
"GE_QE_AnnCURxRandom_yugioh_Round10_4721.npz": "K = 10, L = 0.0, rndFirst = False np.mean(results) = 0.6263366336633663"
}

================== pro_wrestling ==================
{
"GE_QE_AnnCURxKMeans_pro_wrestling_Round10_4834.npz": "K = 10, L = 0.0, rndFirst = False np.mean(results) = 0.6531707317073171",
"GE_QE_AnnCURxRandom_pro_wrestling_Round10_4279.npz": "K = 10, L = 0.0, rndFirst = False np.mean(results) = 0.5895121951219512",
"GE_QE_AnnCURxTop_pro_wrestling_Round10_3002.npz": "K = 10, L = 0.0, rndFirst = False np.mean(results) = 0.44536585365853665",
"GE_QE_AnnCURxCoItem_pro_w

In [79]:
ress = dict()
for dataset in DATASETS:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100
    )

    print("LOADED")
    
    
    GE_QE_s = [
        f for f in os.listdir("./")
        if ("GE_QE_Ann" in f) and (dataset in f)
    ]
    
    ress[dataset] = res = dict()
    
    for GE_QE_i in GE_QE_s:
        print("GE_QE_i = ", GE_QE_i)
        loaded = np.load('./' + GE_QE_i)
        ress[dataset][GE_QE_i] = do_AXN(ctx, loaded, get_Rp_basic, STRIP=0.1, Ks=[5])
    
    del ctx
    gc.collect()




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013
LOADED
GE_QE_i =  GE_QE_AnnCURxCoItem_yugioh_Round10_5617.npz


100%|█████████████████████████████████████████| 101/101 [00:44<00:00,  2.26it/s]


K = 5, L = 0.0, TS = 100, rndFirst = False np.mean(results) = 0.7172277227722772 best = 0.7172277227722772


100%|█████████████████████████████████████████| 101/101 [00:48<00:00,  2.07it/s]


K = 5, L = 0.1, TS = 100, rndFirst = False np.mean(results) = 0.6004950495049505 best = 0.7172277227722772


...

K = 5, L = 0.8, TS = 100, rndFirst = False np.mean(results) = 0.30527777777777776 best = 0.3156944444444445


100%|███████████████████████████████████████████| 72/72 [01:11<00:00,  1.01it/s]


K = 5, L = 0.9, TS = 100, rndFirst = False np.mean(results) = 0.3030555555555556 best = 0.3156944444444445


100%|███████████████████████████████████████████| 72/72 [01:06<00:00,  1.09it/s]

K = 5, L = 1.0, TS = 100, rndFirst = False np.mean(results) = 0.30194444444444446 best = 0.3156944444444445


In [49]:
for dataset in DATASETS:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100
    )

    print("LOADED")
    
    do(ctx, f"{dataset}_Round10")
    
    del ctx
    gc.collect()




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013
LOADED


/tmp/ipykernel_179949/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

np.mean(results), mse, len(results) =  0.5840796460176991 0.12217492 2260
np.mean(results), mse, len(results) =  0.5617472852912143 0.146082 1013
./GE_QE_AnnCURxCoItem_yugioh_Round10_5617.npz


/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


...

/tmp/ipykernel_179949/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

np.mean(results), mse, len(results) =  0.4010132995566814 0.006683798 1579
np.mean(results), mse, len(results) =  0.3356944444444444 0.011730117 720
./GE_QE_AnnCURxCoItem_military_Round10_3357.npz


/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


np.mean(results), mse, len(results) =  0.35898036732108934 0.007244896 1579
np.mean(results), mse, len(results) =  0.29630555555555554 0.010896302 720
./GE_QE_AnnCURxKMeans_military_Round10_2963.npz
np.mean(results), mse, len(results) =  0.24730842305256492 0.010067298 1579
np.mean(results), mse, len(results) =  0.1907222222222222 0.016318377 720
./GE_QE_AnnCURxTop_military_Round10_1907.npz
np.mean(results), mse, len(results) =  0.2969411019632679 0.008513261 1579
np.mean(results), mse, len(results) =  0.24543055555555554 0.013691998 720
./GE_QE_AnnCURxRandom_military_Round10_2454.npz


In [51]:
!ls | grep "GE_QE.*RecSys.*npz"

GE_QE_AnnCURxCoItem_RecSysLT_Round10_6565.npz
GE_QE_AnnCURxCoItem_RecSys_Round10_7197.npz
GE_QE_AnnCURxKMeans_RecSysLT_Round10_613.npz
GE_QE_AnnCURxKMeans_RecSys_Round10_7156.npz
GE_QE_AnnCURxRandom_RecSysLT_Round10_5842.npz
GE_QE_AnnCURxRandom_RecSys_Round10_6743.npz
GE_QE_AnnCURxTop_RecSysLT_Round10_6695.npz
GE_QE_AnnCURxTop_RecSys_Round10_7623.npz


In [52]:
!ls | grep "GE_QE.*npz"

GE_QE_AnnCURxCoItem_doctor_who_Round10_296.npz
GE_QE_AnnCURxCoItem_military_Round10_3357.npz
GE_QE_AnnCURxCoItem_pro_wrestling_Round10_5119.npz
GE_QE_AnnCURxCoItem_RecSysLT_Round10_6565.npz
GE_QE_AnnCURxCoItem_RecSys_Round10_7197.npz
GE_QE_AnnCURxCoItem_star_trek_Round10_3677.npz
GE_QE_AnnCURxCoItem_yugioh_Round10_5617.npz
GE_QE_AnnCURxKMeans_doctor_who_Round10_2489.npz
GE_QE_AnnCURxKMeans_military_Round10_2963.npz
GE_QE_AnnCURxKMeans_pro_wrestling_Round10_4834.npz
GE_QE_AnnCURxKMeans_RecSysLT_Round10_613.npz
GE_QE_AnnCURxKMeans_RecSys_Round10_7156.npz
GE_QE_AnnCURxKMeans_star_trek_Round10_3101.npz
GE_QE_AnnCURxKMeans_yugioh_Round10_5017.npz
GE_QE_AnnCURxRandom_doctor_who_Round10_1919.npz
GE_QE_AnnCURxRandom_military_Round10_2454.npz
GE_QE_AnnCURxRandom_pro_wrestling_Round10_4279.npz
GE_QE_AnnCURxRandom_RecSysLT_Round10_5842.npz
GE_QE_AnnCURxRandom_RecSys_Round10_6743.npz
GE_QE_AnnCURxRandom_star_trek_Round10_2287.npz
GE_QE_AnnCURxRandom_yugioh_Round10_4721.npz
GE_QE_AnnCURxTop_doctor_

# RBE

In [53]:
#tbd - make 

In [95]:
m_ = None
def do(ctx, name, coitem=True, kmeans=True, top=True, random=True, N = 100000, LR = 0.0005):
    def get_save_callback(t, fname):
        def save_callback(m):
            global m_
            m_ = m
            GE_ = m.get_game_embs()
            GE = np.hstack([
                GE_,
                m.g_dssm(GE_),
                m.vb.numpy().reshape((-1, 1)) ,  # Vertical bias
                ctx.get_relevs("train").mean(axis=0).reshape((-1, 1))  # popbias
            ])

            R_All = ctx.get_relevs(t)
            QE_ = m.get_user_embs(t)
            QE = np.hstack([
                QE_ @ m.W,
                m.u_dssm(QE_),
                np.ones((R_All.shape[0], 1)),
                np.ones((R_All.shape[0], 1)) * m.pb
            ])
            
            l, r = fname.split(".npz")
            score = m.get_score(t)
            fname_ = l + f"_{str(round(score, 4))}.npz" + r
            print(f"Saving {fname_} ...")
            np.savez_compressed(fname_, QE=QE, GE=GE)
        return save_callback
            
    if coitem:
        t = ctx.get_relevs("train").T
        t = (t - t.mean()) / t.std()  # like in other comparisons with RBE
        ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

        ma = CBKnnV0(ctx, fit_kwargs={
            'c': 0,
            'train_matrix': True,
            'train_dssm': True,
            'train_vbias': True,
            'train_popbias': True, 'train_bias': True,
            'verbose': False, 'loss': 'softmax_signed',
            'loss_batch': 128, 'loss_q': 0.99,
            # 'dssm_l2': 5e-5,
            'activation': 'elu',
            'n': N,
            # 'ubatch': 1500,
            'score_verbose': 1000,
            'trainable_items': False,
            "TEinit": "anncur",
            "use_keys_in_train": True, # <<< DIFF HERE
            "learning_rate": LR,
            "save_callback": get_save_callback("test", f"./GE_QE_RBExCoItem_{name}.npz")
        })

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")
        print(tr, te)
        
    if kmeans:
        ctx.set_kmeans_games_as_key()
        ma = CBKnnV0(ctx, fit_kwargs={
            'c': 0,
            'train_matrix': True,
            'train_dssm': True,
            'train_vbias': True,
            'train_popbias': True, 'train_bias': True,
            'verbose': False, 'loss': 'softmax_signed',
            'loss_batch': 128, 'loss_q': 0.99,
            # 'dssm_l2': 5e-5,
            'activation': 'elu',
            'n': N,
            # 'ubatch': 1500,
            'score_verbose': 1000,
            'trainable_items': False,
            "TEinit": "anncur",
            "use_keys_in_train": True, # <<< DIFF HERE
            "learning_rate": LR,
            "save_callback": get_save_callback("test", f"./GE_QE_RBExKMeans_{name}.npz")
        })

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")
        print(tr, te)
        
    if top:
        ctx.set_top_games_as_key()
        ma = CBKnnV0(ctx, fit_kwargs={
            'c': 0,
            'train_matrix': True,
            'train_dssm': True,
            'train_vbias': True,
            'train_popbias': True, 'train_bias': True,
            'verbose': False, 'loss': 'softmax_signed',
            'loss_batch': 128, 'loss_q': 0.99,
            # 'dssm_l2': 5e-5,
            'activation': 'elu',
            'n': N,
            # 'ubatch': 1500,
            'score_verbose': 1000,
            'trainable_items': False,
            "TEinit": "anncur",
            "use_keys_in_train": True, # <<< DIFF HERE
            "learning_rate": LR,
            "save_callback": get_save_callback("test", f"./GE_QE_RBExTop_{name}.npz")
        })

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")
        print(tr, te)
        
    if random:
        ma = CBKnnV0(ctx, fit_kwargs={
            'c': 0,
            'train_matrix': True,
            'train_dssm': True,
            'train_vbias': True,
            'train_popbias': True, 'train_bias': True,
            'verbose': False, 'loss': 'softmax_signed',
            'loss_batch': 128, 'loss_q': 0.99,
            # 'dssm_l2': 5e-5,
            'activation': 'elu',
            'n': N,
            # 'ubatch': 1500,
            'score_verbose': 1000,
            'trainable_items': False,
            "TEinit": "anncur",
            "use_keys_in_train": True, # <<< DIFF HERE
            "learning_rate": LR,
            "save_callback": get_save_callback("test", f"./GE_QE_{name}_RBExRandom.npz")
        })

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")
        print(tr, te)

In [96]:
DATASETS = ["pro_wrestling", "yugioh", "star_trek", "doctor_who", "military"]

In [ ]:
for dataset in DATASETS:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100
    )

    print("LOADED")
    
    do(ctx, f"{dataset}_Round10")
    
    del ctx
    gc.collect()




DATASET =  pro_wrestling
Loading file pro_wrestling/ment_to_ent_scores_n_m_392_n_e_10133_all_layers_False1000.pkl
Loading file pro_wrestling/ment_to_ent_scores_n_m_500_n_e_10133_all_layers_False0.pkl
Loading file pro_wrestling/ment_to_ent_scores_n_m_500_n_e_10133_all_layers_False500.pkl
Loaded shape =  (1392, 10133)
Shuffling... (seed = 42)
updated det (1, 5.124810435461604e-33 -> 1.0217253487036523e-31)
updated det (2, 1.0217253487036523e-31 -> 1.2030961336739456e-31)
updated det (11, 1.2030961336739456e-31 -> 7.606045638566237e-29)
updated det (60, 7.606045638566237e-29 -> 3.735096148850118e-27)
Best det =  3.735096e-27
Current de =  3.735096e-27
100 873 418
LOADED


/tmp/ipykernel_179949/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

self.embed_users['train'].shape =  (873, 100)
self.embed_games.shape =  (10133, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 10133)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (873, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0347 tf.Tensor(38.846825, shape=(), dtype=float32) 100
slice = key, score = 0.0347
np.mean(results), mse, len(results) =  0.04063001145475372 tf.Tensor(38.683475, shape=(), dtype=float32) 873
slice = train, score = 0.04063001145475372
np.mean(results), mse, len(results) =  0.038875598086124404 tf.Tensor(39.605145, shape=(), dtype=float32) 418
slice = test, score = 0.038875598086124404
np.mean(results), mse, len(results) =  0.038875598086124404 tf.Tensor(39.605145, shape=(), dtype=float32) 418
Saving ./GE_QE_RBExCoItem_pro_wrestling_Round10_0.0389.npz ...

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.5015 tf.Tensor(96.93215, shape=(), dtype=float32) 100
slic

np.mean(results), mse, len(results) =  0.6090148911798398 tf.Tensor(921.9965, shape=(), dtype=float32) 873
slice = train, score = 0.6090148911798398
np.mean(results), mse, len(results) =  0.5154545454545454 tf.Tensor(972.0343, shape=(), dtype=float32) 418
slice = test, score = 0.5154545454545454

=== Iteration 14000 ===
np.mean(results), mse, len(results) =  0.5649000000000001 tf.Tensor(1024.5127, shape=(), dtype=float32) 100
slice = key, score = 0.5649000000000001
np.mean(results), mse, len(results) =  0.6130011454753722 tf.Tensor(1000.46484, shape=(), dtype=float32) 873
slice = train, score = 0.6130011454753722
np.mean(results), mse, len(results) =  0.5164114832535885 tf.Tensor(1061.1006, shape=(), dtype=float32) 418
slice = test, score = 0.5164114832535885

=== Iteration 15000 ===
np.mean(results), mse, len(results) =  0.5654000000000001 tf.Tensor(1105.9341, shape=(), dtype=float32) 100
slice = key, score = 0.5654000000000001
np.mean(results), mse, len(results) =  0.6127835051546392

...


=== Iteration 20000 ===
np.mean(results), mse, len(results) =  0.4942000000000001 tf.Tensor(1259.36, shape=(), dtype=float32) 100
slice = key, score = 0.4942000000000001
np.mean(results), mse, len(results) =  0.5540778923253149 tf.Tensor(1134.6144, shape=(), dtype=float32) 873
slice = train, score = 0.5540778923253149
np.mean(results), mse, len(results) =  0.31011961722488035 tf.Tensor(1142.7065, shape=(), dtype=float32) 418
slice = test, score = 0.31011961722488035
np.mean(results), mse, len(results) =  0.31011961722488035 tf.Tensor(1142.7065, shape=(), dtype=float32) 418
Saving ./GE_QE_RBExTop_pro_wrestling_Round10_0.3101.npz ...

=== Iteration 21000 ===
np.mean(results), mse, len(results) =  0.5022000000000001 tf.Tensor(1393.2922, shape=(), dtype=float32) 100
slice = key, score = 0.5022000000000001
np.mean(results), mse, len(results) =  0.5573195876288659 tf.Tensor(1220.0471, shape=(), dtype=float32) 873
slice = train, score = 0.5573195876288659
np.mean(results), mse, len(results) 


=== Iteration 34000 ===
np.mean(results), mse, len(results) =  0.5155000000000001 tf.Tensor(2029.6626, shape=(), dtype=float32) 100
slice = key, score = 0.5155000000000001
np.mean(results), mse, len(results) =  0.5776403207331042 tf.Tensor(1827.6936, shape=(), dtype=float32) 873
slice = train, score = 0.5776403207331042
np.mean(results), mse, len(results) =  0.3094976076555024 tf.Tensor(1842.635, shape=(), dtype=float32) 418
slice = test, score = 0.3094976076555024

=== Iteration 35000 ===
np.mean(results), mse, len(results) =  0.5201 tf.Tensor(2119.105, shape=(), dtype=float32) 100
slice = key, score = 0.5201
np.mean(results), mse, len(results) =  0.5852119129438716 tf.Tensor(1876.731, shape=(), dtype=float32) 873
slice = train, score = 0.5852119129438716
np.mean(results), mse, len(results) =  0.31449760765550233 tf.Tensor(1937.3384, shape=(), dtype=float32) 418
slice = test, score = 0.31449760765550233
np.mean(results), mse, len(results) =  0.31449760765550233 tf.Tensor(1937.3384, s

np.mean(results), mse, len(results) =  0.5969186712485681 tf.Tensor(2748.1206, shape=(), dtype=float32) 873
slice = train, score = 0.5969186712485681
np.mean(results), mse, len(results) =  0.3123923444976076 tf.Tensor(2807.8994, shape=(), dtype=float32) 418
slice = test, score = 0.3123923444976076

=== Iteration 50000 ===
np.mean(results), mse, len(results) =  0.5361000000000001 tf.Tensor(3096.9446, shape=(), dtype=float32) 100
slice = key, score = 0.5361000000000001
np.mean(results), mse, len(results) =  0.5976288659793815 tf.Tensor(2841.5115, shape=(), dtype=float32) 873
slice = train, score = 0.5976288659793815
np.mean(results), mse, len(results) =  0.3141387559808613 tf.Tensor(2922.7798, shape=(), dtype=float32) 418
slice = test, score = 0.3141387559808613
np.mean(results), mse, len(results) =  0.3141387559808613 tf.Tensor(2922.7798, shape=(), dtype=float32) 418
Saving ./GE_QE_RBExTop_pro_wrestling_Round10_0.3141.npz ...

=== Iteration 51000 ===
np.mean(results), mse, len(results) 

np.mean(results), mse, len(results) =  0.3087799043062201 tf.Tensor(3714.9763, shape=(), dtype=float32) 418
Saving ./GE_QE_RBExTop_pro_wrestling_Round10_0.3088.npz ...

=== Iteration 66000 ===
np.mean(results), mse, len(results) =  0.5457 tf.Tensor(4151.6064, shape=(), dtype=float32) 100
slice = key, score = 0.5457
np.mean(results), mse, len(results) =  0.6054410080183276 tf.Tensor(3803.5496, shape=(), dtype=float32) 873
slice = train, score = 0.6054410080183276
np.mean(results), mse, len(results) =  0.31332535885167456 tf.Tensor(3921.9827, shape=(), dtype=float32) 418
slice = test, score = 0.31332535885167456

=== Iteration 67000 ===
np.mean(results), mse, len(results) =  0.5435000000000001 tf.Tensor(4157.6426, shape=(), dtype=float32) 100
slice = key, score = 0.5435000000000001
np.mean(results), mse, len(results) =  0.60475372279496 tf.Tensor(3830.0808, shape=(), dtype=float32) 873
slice = train, score = 0.60475372279496
np.mean(results), mse, len(results) =  0.30923444976076553 tf.T


=== Iteration 82000 ===
np.mean(results), mse, len(results) =  0.5555 tf.Tensor(5350.5923, shape=(), dtype=float32) 100
slice = key, score = 0.5555
np.mean(results), mse, len(results) =  0.6156357388316152 tf.Tensor(4952.994, shape=(), dtype=float32) 873
slice = train, score = 0.6156357388316152
np.mean(results), mse, len(results) =  0.3040909090909091 tf.Tensor(5174.693, shape=(), dtype=float32) 418
slice = test, score = 0.3040909090909091
np.mean(results), mse, len(results) =  0.3040909090909091 tf.Tensor(5174.693, shape=(), dtype=float32) 418
Saving ./GE_QE_RBExTop_pro_wrestling_Round10_0.3041.npz ...

=== Iteration 83000 ===
np.mean(results), mse, len(results) =  0.5486000000000001 tf.Tensor(5319.2505, shape=(), dtype=float32) 100
slice = key, score = 0.5486000000000001
np.mean(results), mse, len(results) =  0.609335624284078 tf.Tensor(4944.1753, shape=(), dtype=float32) 873
slice = train, score = 0.609335624284078
np.mean(results), mse, len(results) =  0.311866028708134 tf.Tensor

In [81]:
!ls | grep "GE_QE_RBE"

GE_QE_RBExCoItem_doctor_who_Round10_0.0261.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.2567.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.2672.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.2766.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.2818.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.2831.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.2832.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.2835.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.2854.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.2876.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.2879.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.2908.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.2911.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.2913.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.2915.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.2932.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.2945.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.2946.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.2947.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.294.npz
GE_QE_RBExCoItem_doctor_who_Round10_0.295.npz
GE_QE_RBExCoIte

In [82]:
QE_ = m_.get_user_embs("test")
R_All = ctx.get_relevs("test")
QE = np.hstack([
    QE_ @ m_.W,
    m_.u_dssm(QE_),
    np.ones((R_All.shape[0], 1)),
    np.ones((R_All.shape[0], 1)) * m_.pb
])

In [ ]:
for dataset in DATASETS:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100
    )

    print("LOADED")
    
    do(ctx, f"{dataset}_Round10")
    
    del ctx
    gc.collect()

In [7]:
t = ctx.get_relevs("train").T
t = (t - t.mean()) / t.std()  # like in other comparisons with RBE
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_358619/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.7277067183462532 0.17279571845624642 4644
np.mean(results), mse, len(results) =  0.7196509341199606 0.27000474140976904 2034


(0.7277067183462532, 0.7196509341199606)

In [8]:
GE = ma.get_game_embs()
QE = torch.from_numpy(ma.ctx.get_relevs("test")[:, ma.key_cols_idx])

np.savez_compressed("./GE_QE_AnnCURxCoItem_RecSys_pairwise.npz", QE=QE.numpy(), GE=GE.numpy())

In [10]:
loaded = np.load('./GE_QE_AnnCURxCoItem_RecSys_pairwise.npz')
loaded["GE"], loaded["QE"]

(array([[-6.99894281e-07,  2.28082557e-07, -7.68262684e-07, ...,
          7.21491372e-06, -9.86397304e-06,  2.38271599e-04],
        [ 4.48866242e-06, -7.16671929e-07, -1.00057667e-06, ...,
          3.12123734e-05,  1.01422295e-04,  7.46128584e-04],
        [-6.79450750e-07, -3.54730573e-07, -5.77115023e-07, ...,
          1.04192418e-05, -2.64107717e-07,  3.02313833e-04],
        ...,
        [ 8.76051060e-06, -6.40698277e-05, -5.49386342e-05, ...,
         -2.15575740e-04, -4.94313477e-05,  2.64700002e-02],
        [-1.79433635e-05,  4.32205534e-05,  3.56475006e-05, ...,
         -1.74609501e-05,  2.82109294e-04,  1.29137913e-03],
        [-4.59748938e-05,  5.35147263e-05, -2.80591650e-06, ...,
          6.00601974e-05,  4.03257544e-04,  6.37227699e-03]]),
 array([[ 0.72265023,  0.3253895 ,  1.09422529, ...,  0.72851855,
          0.46345094,  1.75333095],
        [ 2.5486567 ,  0.75778222,  0.54785085, ..., 11.48628902,
         13.26927948,  8.32145691],
        [ 0.77662414,  0.

In [20]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for K in [25]:
        for L in [1]:
            for T, TS in [(500, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                    for _ in range(T // K - 1):
                        Rp = get_Rp(GE, A, Rt[A], QE[i], L)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True or (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:TS]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 101/101 [00:07<00:00, 12.80it/s]

K = 25, L = 1, TS = 100, rndFirst = False np.mean(results) = 0.06683168316831684


'K = 25, L = 1, rndFirst = False np.mean(results) = 0.06683168316831684'

In [21]:
first_of = K
hits = list()
for i in range(len(As)):
    first = As[i][first_of:first_of+K]
    for of_ in range(first_of+K, T, K):
        current = As[i][of_:of_+25]
        hits.append(len(set(current).intersection(set(first))))
        
np.mean(hits), len(hits), T//K

(3.0792079207920793, 1818, 20)

In [23]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for K in [25]:
        for L in [1]:
            for T, TS in [(200, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                    for _ in range(T // K - 1):
                        Rp = get_Rp(GE, A, Rt[A], QE[i], L)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True or (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:TS]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 101/101 [00:02<00:00, 36.27it/s]

K = 25, L = 1, TS = 100, rndFirst = False np.mean(results) = 0.38000000000000017


'K = 25, L = 1, rndFirst = False np.mean(results) = 0.38000000000000017'

In [26]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for K in [25]:
        for L in [1]:
            for T, TS in [(100, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                    for _ in range(T // K - 1):
                        Rp = get_Rp(GE, A, Rt[A], QE[i], L)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if False or (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:TS]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 101/101 [00:01<00:00, 74.62it/s]

K = 25, L = 1, TS = 100, rndFirst = False np.mean(results) = 0.38495049504950496


'K = 25, L = 1, rndFirst = False np.mean(results) = 0.38495049504950496'

In [25]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for K in [25]:
        for L in np.linspace(0, 1, 11):
            for T, TS in [(100, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                    for _ in range(T // K - 1):
                        Rp = get_Rp(GE, A, Rt[A], QE[i], L)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if False or (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:TS]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 101/101 [00:01<00:00, 80.30it/s]


K = 25, L = 0.0, TS = 100, rndFirst = False np.mean(results) = 0.7128712871287128


100%|█████████████████████████████████████████| 101/101 [00:01<00:00, 76.14it/s]


K = 25, L = 0.1, TS = 100, rndFirst = False np.mean(results) = 0.4641584158415842


100%|█████████████████████████████████████████| 101/101 [00:01<00:00, 81.24it/s]


...

100%|█████████████████████████████████████████| 101/101 [00:01<00:00, 70.82it/s]


K = 25, L = 0.9, TS = 100, rndFirst = False np.mean(results) = 0.3865346534653466


100%|█████████████████████████████████████████| 101/101 [00:01<00:00, 73.58it/s]

K = 25, L = 1.0, TS = 100, rndFirst = False np.mean(results) = 0.38495049504950496


'K = 25, L = 0.0, rndFirst = False np.mean(results) = 0.7128712871287128'

In [27]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for K in [25]:
        for L in np.linspace(0, 0.1, 11):
            for T, TS in [(100, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                    for _ in range(T // K - 1):
                        Rp = get_Rp(GE, A, Rt[A], QE[i], L)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if False or (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:TS]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 101/101 [00:02<00:00, 41.11it/s]


K = 25, L = 0.0, TS = 100, rndFirst = False np.mean(results) = 0.7128712871287128


100%|█████████████████████████████████████████| 101/101 [00:01<00:00, 66.79it/s]


K = 25, L = 0.01, TS = 100, rndFirst = False np.mean(results) = 0.608019801980198


100%|█████████████████████████████████████████| 101/101 [00:01<00:00, 58.21it/s]


...

100%|█████████████████████████████████████████| 101/101 [00:01<00:00, 63.33it/s]


K = 25, L = 0.09, TS = 100, rndFirst = False np.mean(results) = 0.46801980198019805


100%|█████████████████████████████████████████| 101/101 [00:01<00:00, 64.23it/s]

K = 25, L = 0.1, TS = 100, rndFirst = False np.mean(results) = 0.4641584158415842


'K = 25, L = 0.0, rndFirst = False np.mean(results) = 0.7128712871287128'

In [25]:
loaded = np.load('./GE_QE_AnnCURxCoItem.npz')
loaded["GE"], loaded["QE"]

(array([[ 0.00474047, -0.00134592,  0.00105135, ..., -0.01403404,
         -0.06027953,  0.03029055],
        [ 0.00304035,  0.00348719,  0.00059625, ..., -0.02397177,
          0.02368937, -0.00214129],
        [-0.00171048, -0.00074191, -0.00312175, ..., -0.02022368,
         -0.04254626,  0.0015299 ],
        ...,
        [-0.00149242, -0.00083971,  0.00127875, ...,  0.30800236,
          0.01279671,  0.31648819],
        [ 0.00138912, -0.00338582, -0.00446363, ...,  0.21003374,
         -0.00506344,  0.06520486],
        [-0.00499681,  0.0055155 ,  0.00054058, ...,  0.34524469,
          0.0484674 ,  0.09927237]]),
 array([[ 6.1222043 ,  3.87819266,  5.50115252, ...,  4.24813557,
          4.1863637 ,  5.0663023 ],
        [11.11496925,  7.97319746,  8.55467796, ...,  6.13713265,
          6.86200666,  7.77079678],
        [10.17910194,  8.9642086 , 10.32344437, ...,  5.2055912 ,
          6.34264612,  5.86023664],
        ...,
        [ 8.3173008 ,  7.88316393, 11.69620609, ...,  

In [12]:
import numpy as np


def get_Rp(GE, A, R, QE_i, L):
    Ae = GE[A]

    u = np.linalg.pinv(Ae.T @ Ae) @ Ae.T @ R
    assert u.shape == QE_i.shape
    Rp = GE @ (u * L + QE_i * (1 - L))
    
    return Rp

from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for K in [25]:
        for L in [1]:
            for T, TS in [(500, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                    for _ in range(T // K - 1):
                        Rp = get_Rp(GE, A, Rt[A], QE[i], L)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            #if True and (ai not in An):
                            An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:TS]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 101/101 [00:10<00:00, 10.07it/s]

K = 25, L = 1, TS = 100, rndFirst = False np.mean(results) = 0.06683168316831684


'K = 25, L = 1, rndFirst = False np.mean(results) = 0.06683168316831684'

In [39]:
first_of = K
hits = list()
for i in range(len(As)):
    first = As[i][first_of:first_of+K]
    for of_ in range(first_of+K, T, K):
        current = As[i][of_:of_+25]
        hits.append(len(set(current).intersection(set(first))))
        
np.mean(hits), len(hits), T//K

(17.59188034188034, 1872, 20)

In [40]:
hits

[23,
 22,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 4,
 4,
 4,
 5,
 4,
 0,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 17,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 9,
 9,
 10,
 10,
 10,
 10,
 10,
 10,
 10,
 10,
 10,
 10,
 10,
 10,
 10,
 10,
 10,
 10,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 15,
 15,
 15,
 15,
 15,
 15,
 15,
 15,
 15,
 15,
 15,
 15,
 15,
 15,
 15,
 15,
 15,
 15,
 22,
 21,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 19,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 17,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 17,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,


In [27]:
import numpy as np


def get_Rp(GE, A, R, QE_i, L):
    Ae = GE[A]

    u = np.linalg.pinv(Ae.T @ Ae) @ Ae.T @ R
    assert u.shape == QE_i.shape
    Rp = GE @ (u * L + QE_i * (1 - L))
    
    return Rp

In [41]:
loaded = np.load('./GE_QE_DSSM8_at_100k.npz')
loaded["GE"], loaded["QE"]

(array([[ 2.97130000e-02, -3.27346000e-02, -3.46989000e-02, ...,
         -7.82296479e-01, -4.97554149e-03,  5.33902600e+00],
        [ 2.02651000e-02,  7.19004000e-02,  2.23142000e-03, ...,
          1.61782712e-01, -2.00821459e-02,  5.53517628e+00],
        [-1.20589000e-02,  1.87944000e-02,  4.56247000e-02, ...,
         -1.30174786e-01, -1.23924620e-01,  5.32502247e+00],
        ...,
        [-2.35462000e-02,  1.20831000e-02, -4.17760000e-02, ...,
         -1.03709483e+00, -3.64314020e-02,  5.77182733e+00],
        [ 1.51869000e-03,  1.35546000e-01, -1.71226000e-02, ...,
         -8.92335832e-01, -3.33648175e-03,  5.12969962e+00],
        [ 2.19628000e-02,  1.30364000e-01, -2.18127000e-02, ...,
         -4.20807838e-01, -2.88077407e-02,  5.08200507e+00]]),
 array([[-0.8176709 , -0.47645564,  1.50424184, ..., -1.13001204,
          1.        , 13.40936565],
        [-0.68234849, -0.7408109 ,  1.5917183 , ..., -0.94634646,
          1.        , 13.40936565],
        [-0.56474965, -0.

In [62]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for K in [25]:
        for L in [0.2]:
            for T, TS in [(500, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                    for _ in range(T // K - 1):
                        Rp = get_Rp(GE, A, Rt[A], QE[i], L)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True and (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:TS]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 104/104 [00:27<00:00,  3.81it/s]

K = 25, L = 0.2, TS = 100, rndFirst = False np.mean(results) = 0.8960576923076923


'K = 25, L = 0.2, rndFirst = False np.mean(results) = 0.8960576923076923'

In [76]:
first_of = 18*K
hits = list()
for i in range(len(As)):
    first = As[i][first_of:first_of+K]
    for of_ in range(first_of+K, T, K):
        current = As[i][of_:of_+25]
        hits.append(len(set(current).intersection(set(first))))

In [77]:
np.mean(hits), len(hits)

(0.0, 104)

In [73]:
As[0][-50:]

array([  585, 12260, 15191,  9395,  2354,  6938, 13644,  2023, 14955,
         384,  8178,  8164,  9807,  8660,   484, 14362, 12230,  8931,
       12834, 13491, 16289, 14568, 14113,   883, 16508,  9127,  1485,
       15243, 13757,  3698, 16497,  9419, 15351, 13654,  4116, 15864,
        4727,  1140,  2285, 14844, 15522, 13979,  5764,  7331, 16004,
        6086, 13556, 10202,  6585, 10757])

In [78]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for K in [25]:
        for L in [1]:
            for T, TS in [(500, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                    for _ in range(T // K - 1):
                        Rp = get_Rp(GE, A, Rt[A], QE[i], L)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True and (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:TS]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 104/104 [00:27<00:00,  3.73it/s]

K = 25, L = 1, TS = 100, rndFirst = False np.mean(results) = 0.8254807692307693


'K = 25, L = 1, rndFirst = False np.mean(results) = 0.8254807692307693'

In [85]:
first_of = 18*K
hits = list()
for i in range(len(As)):
    first = As[i][first_of:first_of+K]
    for of_ in range(first_of+K, T, K):
        current = As[i][of_:of_+25]
        hits.append(len(set(current).intersection(set(first))))
        
np.mean(hits), len(hits), T//K

(0.0, 104, 20)

In [86]:
first

array([12617,  7448, 13282, 10950,  6536,  6421, 10730, 12642,  2271,
        6072,  6670,  2945,  2354,  9318,  6223,  7032, 11479, 10513,
       11782,  3450,  1819,  3286,  1209,  1817,  3433])

# Momentum

In [43]:
import numpy as np


def get_Rp_last(GE, A, R, QE_i, L, u_last, m=0.5):
    Ae = GE[A]

    u = np.linalg.pinv(Ae.T @ Ae) @ Ae.T @ R
    assert u.shape == QE_i.shape

    if u_last is not None:
        u = (u * (1 - m) + u_last * m)

    Rp = GE @ (u * L + QE_i * (1 - L))
    
    return Rp, u

In [45]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for K in [25]:
        for L in [1]:
            for T, TS in [(500, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
                    
                    u_last = None
                    for _ in range(T // K - 1):
                        Rp, u_last = get_Rp_last(GE, A, Rt[A], QE[i], L, u_last, m=0.5)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True and (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:TS]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 104/104 [00:24<00:00,  4.29it/s]

K = 25, L = 1, TS = 100, rndFirst = False np.mean(results) = 0.8231730769230767


'K = 25, L = 1, rndFirst = False np.mean(results) = 0.8231730769230767'

In [46]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for K in [25]:
        for L in [1]:
            for T, TS in [(500, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
                    
                    u_last = QE[i]
                    for _ in range(T // K - 1):
                        Rp, u_last = get_Rp_last(GE, A, Rt[A], QE[i], L, u_last, m=0.5)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True and (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:TS]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 104/104 [00:24<00:00,  4.23it/s]

K = 25, L = 1, TS = 100, rndFirst = False np.mean(results) = 0.8463461538461539


'K = 25, L = 1, rndFirst = False np.mean(results) = 0.8463461538461539'

In [50]:
first_of = 18*K
hits = list()
for i in range(len(As)):
    first = As[i][first_of:first_of+K]
    for of_ in range(first_of+K, T, K):
        current = As[i][of_:of_+25]
        hits.append(len(set(current).intersection(set(first))))
        
np.mean(hits), len(hits), T//K

(0.0, 104, 20)

In [51]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for K in [25]:
        for L in [1]:
            for T, TS in [(500, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
                    
                    u_last = QE[i]
                    for _ in range(T // K - 1):
                        Rp, u_last = get_Rp_last(GE, A, Rt[A], QE[i], L, u_last, m=0.8)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True and (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:TS]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 104/104 [00:22<00:00,  4.57it/s]

K = 25, L = 1, TS = 100, rndFirst = False np.mean(results) = 0.8648076923076923


'K = 25, L = 1, rndFirst = False np.mean(results) = 0.8648076923076923'

In [52]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for K in [25]:
        for L in [1]:
            for T, TS in [(500, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
                    
                    u_last = QE[i]
                    for _ in range(T // K - 1):
                        Rp, u_last = get_Rp_last(GE, A, Rt[A], QE[i], L, u_last, m=1)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True and (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:TS]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 104/104 [00:24<00:00,  4.23it/s]

K = 25, L = 1, TS = 100, rndFirst = False np.mean(results) = 0.893846153846154


'K = 25, L = 1, rndFirst = False np.mean(results) = 0.893846153846154'

In [53]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for K in [25]:
        for L in [0.2]:
            for T, TS in [(200, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
                    
                    u_last = QE[i]
                    for _ in range(T // K - 1):
                        Rp, u_last = get_Rp_last(GE, A, Rt[A], QE[i], L, u_last, m=0)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True and (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:TS]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 104/104 [00:07<00:00, 13.29it/s]

K = 25, L = 0.2, TS = 100, rndFirst = False np.mean(results) = 0.720096153846154


'K = 25, L = 0.2, rndFirst = False np.mean(results) = 0.720096153846154'

In [54]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for K in [25]:
        for L in [0.2]:
            for T, TS in [(200, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
                    
                    u_last = QE[i]
                    for _ in range(T // K - 1):
                        Rp, u_last = get_Rp_last(GE, A, Rt[A], QE[i], L, u_last, m=0.5)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True and (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:TS]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 104/104 [00:07<00:00, 14.17it/s]

K = 25, L = 0.2, TS = 100, rndFirst = False np.mean(results) = 0.7191346153846154


'K = 25, L = 0.2, rndFirst = False np.mean(results) = 0.7191346153846154'

In [55]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for K in [25]:
        for L in [0.2]:
            for T, TS in [(200, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
                    
                    u_last = None
                    for _ in range(T // K - 1):
                        Rp, u_last = get_Rp_last(GE, A, Rt[A], QE[i], L, u_last, m=0.5)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True and (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:TS]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 104/104 [00:06<00:00, 15.57it/s]

K = 25, L = 0.2, TS = 100, rndFirst = False np.mean(results) = 0.7196153846153847


'K = 25, L = 0.2, rndFirst = False np.mean(results) = 0.7196153846153847'

In [56]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for K in [25]:
        for L in [0.2]:
            for T, TS in [(200, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
                    
                    u_last = None
                    for _ in range(T // K - 1):
                        Rp, u_last = get_Rp_last(GE, A, Rt[A], QE[i], L, u_last, m=0.25)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True and (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:TS]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 104/104 [00:06<00:00, 15.30it/s]

K = 25, L = 0.2, TS = 100, rndFirst = False np.mean(results) = 0.7185576923076922


'K = 25, L = 0.2, rndFirst = False np.mean(results) = 0.7185576923076922'

In [58]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for K in [25]:
        for L in [0.5]:
            for T, TS in [(200, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
                    
                    u_last = None
                    for _ in range(T // K - 1):
                        Rp, u_last = get_Rp_last(GE, A, Rt[A], QE[i], L, u_last, m=0.5)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True and (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:TS]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 104/104 [00:06<00:00, 16.33it/s]

K = 25, L = 0.5, TS = 100, rndFirst = False np.mean(results) = 0.7146153846153845


'K = 25, L = 0.5, rndFirst = False np.mean(results) = 0.7146153846153845'

In [59]:
import numpy as np


def get_Rp_last(GE, A, R, QE_i, L, u_last, m=0.5):
    Ae = GE[A]

    u = np.linalg.pinv(Ae.T @ Ae) @ Ae.T @ R
    assert u.shape == QE_i.shape

    if u_last is not None:
        u = (u * (1 - m) + u_last * m)

    u = u * L + QE_i * (1 - L)
    Rp = GE @ u
    
    return Rp, u

In [62]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for K in [25]:
        for L in [1]:
            for T, TS in [(200, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
                    
                    u_last = QE[i]
                    for _ in range(T // K - 1):
                        Rp, u_last = get_Rp_last(GE, A, Rt[A], QE[i], L, u_last, m=0.9)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True and (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:TS]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 104/104 [00:08<00:00, 12.29it/s]

K = 25, L = 1, TS = 100, rndFirst = False np.mean(results) = 0.7204807692307692


'K = 25, L = 1, rndFirst = False np.mean(results) = 0.7204807692307692'

In [63]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for K in [25]:
        for L in [1]:
            for T, TS in [(200, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
                    
                    u_last = QE[i]
                    for _ in range(T // K - 1):
                        Rp, u_last = get_Rp_last(GE, A, Rt[A], QE[i], L, u_last, m=0.95)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True and (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:TS]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 104/104 [00:08<00:00, 12.50it/s]

K = 25, L = 1, TS = 100, rndFirst = False np.mean(results) = 0.7202884615384615


'K = 25, L = 1, rndFirst = False np.mean(results) = 0.7202884615384615'

In [70]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for m in np.linspace(0, 1, 11):
        for K in [25]:
            for L in np.linspace(0, 1, 11):
                for T, TS in [(200, 100)]:
                    results = list()


                    As = list()
                    for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                        A = (
                            ctx.key_games[:K]
                            if randomFirst else
                            (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                        )
                        Rt = R_All[i]
                        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                        u_last = QE[i]
                        for _ in range(T // K - 1):
                            Rp, u_last = get_Rp_last(GE, A, Rt[A], QE[i], L, u_last, m=m)

                            An = list(A)
                            for ai in Rp.argsort()[::-1]:
                                if True and (ai not in An):
                                    An.append(ai)
                                if len(An) == len(A) + K:
                                    break
                            A = np.array(An)

                        assert len(A) == T
                        As.append(A)


                        A = sorted([
                            (-Rt[ai], ai)
                            for ai in A
                        ])[:TS]
                        A = [ai for _, ai in A]

                        pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                        results.append(ev(pred_top, gt_top) / float(TS))

                    if np.mean(results) > best:
                        best = np.mean(results)
                        best_arg = f"K = {K}, L = {L}, m = {m}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                        best_arg_d = {
                            "K": K,
                            "L": L,
                            "randomFirst": randomFirst,
                            "score": np.mean(results),
                            "m": m
                        }
                    print(f"K = {K}, L = {L}, m = {m}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")

best_arg

100%|█████████████████████████████████████████| 104/104 [00:08<00:00, 12.71it/s]


K = 25, L = 0.0, m = 0.0, TS = 100, rndFirst = False np.mean(results) = 0.7164423076923077


100%|█████████████████████████████████████████| 104/104 [00:10<00:00,  9.91it/s]


K = 25, L = 0.1, m = 0.0, TS = 100, rndFirst = False np.mean(results) = 0.7189423076923076


100%|█████████████████████████████████████████| 104/104 [00:09<00:00, 10.44it/s]


...

100%|█████████████████████████████████████████| 104/104 [00:07<00:00, 13.48it/s]


K = 25, L = 0.9, m = 1.0, TS = 100, rndFirst = False np.mean(results) = 0.7164423076923077


100%|█████████████████████████████████████████| 104/104 [00:07<00:00, 13.17it/s]

K = 25, L = 1.0, m = 1.0, TS = 100, rndFirst = False np.mean(results) = 0.7164423076923077


'K = 25, L = 0.30000000000000004, m = 0.30000000000000004, rndFirst = False np.mean(results) = 0.720576923076923'

array([15.028804  , 17.60270343, 15.44045345, ..., 15.56038531,
       16.02630586, 14.44371887])

In [102]:
import numpy as np


def get_Rp_best(GE, A, R, QE_i, L, u_last, m=0.5):
    u = (GE[A].T @ (R  / R.sum()).T
    u = u_last * m + u * (1 - m)
    return GE @ u, u
    
    u_mx = GE[A[R.argmax()]]
    u_mn = GE[A[R.argmin()]]
    u = u_last * m + (u_mx - u_mn) * (1 - m)
    return GE @ u, u
    

In [103]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]
#GE = (GE.T / (GE ** 2).sum(axis=1) ** 0.5).T
#QE = (QE.T / (QE ** 2).sum(axis=1) ** 0.5).T
#doesnt work

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for m in np.linspace(0, 1, 11):
        for K in [25]:
            for L in [0]:
                for T, TS in [(200, 100)]:
                    results = list()


                    As = list()
                    for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                        A = (
                            ctx.key_games[:K]
                            if randomFirst else
                            (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                        )
                        Rt = R_All[i]
                        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                        u_last = QE[i]
                        for _ in range(T // K - 1):
                            Rp, u_last = get_Rp_best(GE, A, Rt[A], QE[i], L, u_last, m=m)

                            An = list(A)
                            for ai in Rp.argsort()[::-1]:
                                if True and (ai not in An):
                                    An.append(ai)
                                if len(An) == len(A) + K:
                                    break
                            A = np.array(An)

                        assert len(A) == T
                        As.append(A)


                        A = sorted([
                            (-Rt[ai], ai)
                            for ai in A
                        ])[:TS]
                        A = [ai for _, ai in A]

                        pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                        results.append(ev(pred_top, gt_top) / float(TS))

                    if np.mean(results) > best:
                        best = np.mean(results)
                        best_arg = f"K = {K}, L = {L}, m = {m}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                        best_arg_d = {
                            "K": K,
                            "L": L,
                            "randomFirst": randomFirst,
                            "score": np.mean(results),
                            "m": m
                        }
                    print(f"K = {K}, L = {L}, m = {m}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")

best_arg

100%|█████████████████████████████████████████| 104/104 [00:01<00:00, 54.37it/s]


K = 25, L = 0, m = 0.0, TS = 100, rndFirst = False np.mean(results) = 0.3382692307692308


100%|█████████████████████████████████████████| 104/104 [00:01<00:00, 55.36it/s]


K = 25, L = 0, m = 0.1, TS = 100, rndFirst = False np.mean(results) = 0.3452884615384615


100%|█████████████████████████████████████████| 104/104 [00:01<00:00, 56.05it/s]


...

100%|█████████████████████████████████████████| 104/104 [00:01<00:00, 55.73it/s]


K = 25, L = 0, m = 0.9, TS = 100, rndFirst = False np.mean(results) = 0.5708653846153846


100%|█████████████████████████████████████████| 104/104 [00:01<00:00, 54.65it/s]

K = 25, L = 0, m = 1.0, TS = 100, rndFirst = False np.mean(results) = 0.7164423076923077


'K = 25, L = 0, m = 1.0, rndFirst = False np.mean(results) = 0.7164423076923077'

In [110]:
import numpy as np


def get_Rp_rnd(GE, A, R, QE_i, L, u_last, m=0.5):
    u = QE_i + np.random.rand(GE.shape[1]) * m
    return GE @ u, u
    

In [111]:
from sklearn.linear_model import Ridge

Z = "test"
q = ctx.get_requests(Z)
R_All = ctx.get_relevs(Z)

GE, QE = loaded["GE"], loaded["QE"]
#GE = (GE.T / (GE ** 2).sum(axis=1) ** 0.5).T
#QE = (QE.T / (QE ** 2).sum(axis=1) ** 0.5).T
#doesnt work

best = 0
best_arg = None

STRIP = 0.05

for randomFirst in [False]:
    for m in np.linspace(0, 1, 11):
        for K in [25]:
            for L in [0]:
                for T, TS in [(200, 100)]:
                    results = list()


                    As = list()
                    for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                        A = (
                            ctx.key_games[:K]
                            if randomFirst else
                            (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                        )
                        Rt = R_All[i]
                        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                        u_last = QE[i]
                        for _ in range(T // K - 1):
                            Rp, u_last = get_Rp_rnd(GE, A, Rt[A], QE[i], L, u_last, m=m)

                            An = list(A)
                            for ai in Rp.argsort()[::-1]:
                                if True and (ai not in An):
                                    An.append(ai)
                                if len(An) == len(A) + K:
                                    break
                            A = np.array(An)

                        assert len(A) == T
                        As.append(A)


                        A = sorted([
                            (-Rt[ai], ai)
                            for ai in A
                        ])[:TS]
                        A = [ai for _, ai in A]

                        pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                        results.append(ev(pred_top, gt_top) / float(TS))

                    if np.mean(results) > best:
                        best = np.mean(results)
                        best_arg = f"K = {K}, L = {L}, m = {m}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                        best_arg_d = {
                            "K": K,
                            "L": L,
                            "randomFirst": randomFirst,
                            "score": np.mean(results),
                            "m": m
                        }
                    print(f"K = {K}, L = {L}, m = {m}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")

best_arg

100%|█████████████████████████████████████████| 104/104 [00:02<00:00, 50.28it/s]


K = 25, L = 0, m = 0.0, TS = 100, rndFirst = False np.mean(results) = 0.7164423076923077


100%|█████████████████████████████████████████| 104/104 [00:01<00:00, 59.55it/s]


K = 25, L = 0, m = 0.1, TS = 100, rndFirst = False np.mean(results) = 0.7152884615384614


100%|█████████████████████████████████████████| 104/104 [00:01<00:00, 66.25it/s]


...

100%|█████████████████████████████████████████| 104/104 [00:02<00:00, 48.97it/s]


K = 25, L = 0, m = 0.9, TS = 100, rndFirst = False np.mean(results) = 0.708076923076923


100%|█████████████████████████████████████████| 104/104 [00:02<00:00, 50.61it/s]

K = 25, L = 0, m = 1.0, TS = 100, rndFirst = False np.mean(results) = 0.7087500000000001


'K = 25, L = 0, m = 0.0, rndFirst = False np.mean(results) = 0.7164423076923077'